# BASELINE & MODEL

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

print("===== LOADING DATASET & TOKENIZER =====")

gsm8k = load_dataset("openai/gsm8k", "main")

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-Math-1.5B"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("GSM8K loaded successfully.")
print("Train samples:", len(gsm8k["train"]))
print("Test samples:", len(gsm8k["test"]))
print("Tokenizer loaded successfully.")

===== LOADING DATASET & TOKENIZER =====


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (4 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

GSM8K loaded successfully.
Train samples: 7473
Test samples: 1319
Tokenizer loaded successfully.


In [ ]:
import re

def extract_gsm8k_answer(text):
    """
    Extract final numerical answer from GSM8K-style model output.
    """

    if text is None:
        return None

    # 1. GSM8K format: #### 18
    match = re.search(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        text
    )

    if match:
        return match.group(1)

    # 2. LaTeX boxed format: \boxed{18}
    match = re.search(
        r"\\boxed\{([-+]?\d+(?:\.\d+)?)\}",
        text
    )

    if match:
        return match.group(1)

    return None


def normalize_answer(answer):
    if answer is None:
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")

    # 18.0 -> 18
    if answer.endswith(".0"):
        answer = answer[:-2]

    return answer


def is_correct(predicted, gold):
    predicted = normalize_answer(predicted)
    gold = normalize_answer(gold)

    return predicted == gold

In [ ]:
gold = extract_gsm8k_answer(gsm8k["test"][0]["answer"])

print("Gold answer:", gold)

Gold answer: 18


In [ ]:
print("Test 1:", is_correct("18", "18"))
print("Test 2:", is_correct("18.0", "18"))
print("Test 3:", is_correct("17", "18"))

Test 1: True
Test 2: True
Test 3: False


In [ ]:
def generate_answer(question, max_new_tokens=512):
    prompt = (
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        f"{question}\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [ ]:
from transformers import AutoModelForCausalLM

print("===== LOADING BASE MODEL =====")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Math-1.5B",
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Base model loaded successfully.")
print("Model type:", type(model))
print("Device:", model.device)

===== LOADING BASE MODEL =====


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base model loaded successfully.
Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Device: cuda:0


In [ ]:
question = gsm8k["test"][0]["question"]

response = generate_answer(question)

print(response)

To determine how much Janet makes every day at the farmers' market, we need to follow these steps:

1. Calculate the number of eggs that are left after accounting for the eggs she eats and the eggs she bakes.
2. Determine how many of these remaining eggs are sold at the farmers' market.
3. Calculate the total revenue from selling these eggs.

Let's break it down step by step:

1. Janet's ducks lay 16 eggs per day.
2. She eats 3 eggs for breakfast every morning, so she eats \(3 \times 7 = 21\) eggs in a week (assuming a 7-day week).
3. She bakes 4 eggs for her friends every day, so she bakes \(4 \times 7 = 28\) eggs in a week.
4. The total number of eggs she uses in a week is \(21 + 28 = 49\) eggs.
5. The number of eggs left after using 49 eggs is \(16 \times 7 - 49 = 112 - 49 = 63\) eggs.
6. Since she sells the remaining eggs at the farmers' market, and she sells them at $2 per egg, the total revenue is \(63 \times 2 = 126\) dollars.

Let's confirm this with Python code.
```python
# Co

In [ ]:
predicted = extract_gsm8k_answer(response)
gold = extract_gsm8k_answer(gsm8k["test"][0]["answer"])

print("Predicted:", predicted)
print("Gold:", gold)
print("Correct:", is_correct(predicted, gold))

Predicted: 126
Gold: 18
Correct: False


In [ ]:
import time

results = []

for i in range(10):
    question = gsm8k["test"][i]["question"]
    gold = extract_gsm8k_answer(gsm8k["test"][i]["answer"])

    start_time = time.time()

    response = generate_answer(question)

    latency = time.time() - start_time

    predicted = extract_gsm8k_answer(response)
    correct = is_correct(predicted, gold)

    # Số token model thực sự sinh ra
    inputs = tokenizer(
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        + question,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[1]

    output_tokens = tokenizer(
        response,
        return_tensors="pt"
    )["input_ids"].shape[1]

    results.append({
        "id": i,
        "predicted": predicted,
        "gold": gold,
        "correct": correct,
        "latency": latency,
        "output_tokens": output_tokens
    })

    print(
        f"[{i+1}/10] "
        f"Predicted={predicted} | "
        f"Gold={gold} | "
        f"Correct={correct} | "
        f"Latency={latency:.2f}s | "
        f"Tokens={output_tokens}"
    )

[1/10] Predicted=126 | Gold=18 | Correct=False | Latency=22.76s | Tokens=512
[2/10] Predicted=3 | Gold=3 | Correct=True | Latency=7.71s | Tokens=192
[3/10] Predicted=70000 | Gold=70000 | Correct=True | Latency=20.00s | Tokens=384
[4/10] Predicted=180 | Gold=540 | Correct=False | Latency=6.86s | Tokens=171
[5/10] Predicted=140 | Gold=20 | Correct=False | Latency=16.52s | Tokens=413
[6/10] Predicted=64 | Gold=64 | Correct=True | Latency=8.76s | Tokens=215
[7/10] Predicted=260 | Gold=260 | Correct=True | Latency=10.62s | Tokens=263
[8/10] Predicted=None | Gold=160 | Correct=False | Latency=20.47s | Tokens=512
[9/10] Predicted=None | Gold=45 | Correct=False | Latency=20.65s | Tokens=512
[10/10] Predicted=460 | Gold=460 | Correct=True | Latency=12.87s | Tokens=319


In [ ]:
correct_count = sum(r["correct"] for r in results)
total_count = len(results)

accuracy = correct_count / total_count * 100
avg_latency = sum(r["latency"] for r in results) / total_count
avg_tokens = sum(r["output_tokens"] for r in results) / total_count

print("===== BASELINE 10 SAMPLES =====")
print(f"Correct: {correct_count}/{total_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Average latency: {avg_latency:.2f}s")
print(f"Average generated tokens: {avg_tokens:.2f}")

===== BASELINE 10 SAMPLES =====
Correct: 5/10
Accuracy: 50.00%
Average latency: 14.72s
Average generated tokens: 349.30


# SFT-Cot BASELINE

In [ ]:
train_sample = gsm8k["train"][0]

print("===== QUESTION =====")
print(train_sample["question"])

print("\n===== SOLUTION =====")
print(train_sample["answer"])

===== QUESTION =====
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

===== SOLUTION =====
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [ ]:
def format_sft_example(example):
    return {
        "prompt": (
            "Solve this problem step by step.\n\n"
            f"Problem:\n{example['question']}\n\n"
            "Solution:\n"
        ),
        "completion": example["answer"],
    }

sft_train = gsm8k["train"].map(format_sft_example)

print("SFT prompt/completion dataset created successfully.")
print(sft_train[0])

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

SFT prompt/completion dataset created successfully.
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'prompt': 'Solve this problem step by step.\n\nProblem:\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\n\nSolution:\n', 'completion': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [ ]:
sft_debug = sft_train.select(range(100))

print("Number of SFT samples:", len(sft_debug))
print("\n===== SAMPLE 0 =====")
print("PROMPT:")
print(sft_debug[0]["prompt"])
print("\nCOMPLETION:")
print(sft_debug[0]["completion"])

Number of SFT samples: 100

===== SAMPLE 0 =====
PROMPT:
Solve this problem step by step.

Problem:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Solution:


COMPLETION:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [ ]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.1 MB/s eta 0:00:00


In [ ]:
!pip install -q trl peft
import trl
import peft

print("TRL version:", trl.__version__)
print("PEFT version:", peft.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 21.1 MB/s eta 0:00:00
TRL version: 1.16.0
PEFT version: 0.21.2


In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

print("===== SFT-CoT CONFIG — STABLE VERSION =====")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="./qwen_sft_debug",
    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-6,
    warmup_steps=2,

    logging_steps=5,
    save_strategy="no",
    report_to="none",

    max_length=1024,
    completion_only_loss=True,

    fp16=True,
    gradient_checkpointing=True,

    max_grad_norm=1.0,
)

print("LoRA configuration:")
print("  r =", lora_config.r)
print("  alpha =", lora_config.lora_alpha)
print("  target_modules =", lora_config.target_modules)

print("\nSFT configuration:")
print("  learning_rate =", sft_config.learning_rate)
print("  max_length =", sft_config.max_length)
print("  epochs =", sft_config.num_train_epochs)
print("  batch_size =", sft_config.per_device_train_batch_size)
print("  gradient_accumulation =", sft_config.gradient_accumulation_steps)

print("\nSFT-CoT configuration created successfully.")

===== SFT-CoT CONFIG — STABLE VERSION =====
LoRA configuration:
  r = 8
  alpha = 16
  target_modules = {'v_proj', 'q_proj'}

SFT configuration:
  learning_rate = 2e-06
  max_length = 1024
  epochs = 1
  batch_size = 1
  gradient_accumulation = 4

SFT-CoT configuration created successfully.


In [ ]:
from transformers import AutoModelForCausalLM

print("===== RELOADING BASE MODEL =====")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Math-1.5B",
    torch_dtype="auto",
    device_map="auto"
)

print("Base model reloaded successfully.")
print("Model type:", type(model))

model.forward = model.forward.__func__.__get__(
    model,
    type(model)
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_debug,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("SFTTrainer created successfully.")

===== RELOADING BASE MODEL =====


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model reloaded successfully.
Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

SFTTrainer created successfully.


In [ ]:
print("===== START SFT DEBUG TRAINING =====")
train_result = trainer.train()

print("\n===== SFT DEBUG TRAINING FINISHED =====")
print(train_result)

===== START SFT DEBUG TRAINING =====


Step,Training Loss
5,0.850254
10,0.916669
15,0.850066
20,0.788477
25,0.878481



===== SFT DEBUG TRAINING FINISHED =====
TrainOutput(global_step=25, training_loss=0.8567892837524415, metrics={'train_runtime': 58.4517, 'train_samples_per_second': 1.711, 'train_steps_per_second': 0.428, 'total_flos': 156915239589888.0, 'train_loss': 0.8567892837524415, 'epoch': 1.0})


In [ ]:
import re

print("===== FINAL SFT-CoT TEST =====")

sft_model = trainer.model
sft_model.eval()

print("Model type:", type(sft_model))

print("Has PEFT config:", hasattr(sft_model, "peft_config"))

if hasattr(sft_model, "peft_config"):
    print("PEFT adapters:", list(sft_model.peft_config.keys()))

trainable = 0
total = 0

for name, param in sft_model.named_parameters():
    total += param.numel()

    if param.requires_grad:
        trainable += param.numel()

print("Trainable:", trainable)
print("Total:", total)
print("Trainable %:", 100 * trainable / total)

example = gsm8k["test"][0]

prompt = (
    "Solve this problem step by step.\n\n"
    f"Problem:\n{example['question']}\n\n"
    "Solution:\n"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(sft_model.device)

with torch.no_grad():
    outputs = sft_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

prediction = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

# Extract answer:
# 1. Ưu tiên format GSM8K #### answer
# 2. Nếu không có ####, lấy số cuối cùng trong output
def extract_sft_answer(text):
    match = re.search(r"####\s*([-+]?\d+(?:\.\d+)?)", text)

    if match:
        return match.group(1)

    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", text)

    if numbers:
        return numbers[-1]

    return None


predicted = extract_sft_answer(prediction)
gold = extract_gsm8k_answer(example["answer"])

print("\n===== QUESTION =====")
print(example["question"])

print("\n===== GOLD SOLUTION =====")
print(example["answer"])

print("\n===== SFT-CoT OUTPUT =====")
print(prediction)

print("\n===== ANSWER CHECK =====")
print("Predicted:", predicted)
print("Gold:", gold)
print("Correct:", predicted == gold)

print("\n===== OUTPUT TOKENS =====")
print(len(generated_tokens))

===== FINAL SFT-CoT TEST =====
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Has PEFT config: True
PEFT adapters: ['default']
Trainable: 1089536
Total: 1544803840
Trainable %: 0.07052908413277896

===== QUESTION =====
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

===== GOLD SOLUTION =====
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

===== SFT-CoT OUTPUT =====
First, let's find out how many eggs Janet has left after eating and baking for each day:

Eggs laid per day: 16
Eggs eaten for breakfast: -3
Muffins baked for friends: -4

Remaining eggs per day = 16 - 3 - 4 = 9

Now, let's calculate how much money Janet makes from selling these remaining eggs:

Price 

In [ ]:
print("===== SFT-CoT BENCHMARK — 10 SAMPLES =====")

import time
import re

sft_model = trainer.model
sft_model.eval()

results_sft = []

for i in range(10):

    question = gsm8k["test"][i]["question"]
    gold_answer = extract_gsm8k_answer(
        gsm8k["test"][i]["answer"]
    )

    prompt = (
        "Solve this problem step by step.\n\n"
        f"Problem:\n{question}\n\n"
        "Solution:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(sft_model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = sft_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    latency = time.time() - start_time

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Ưu tiên #### answer
    hash_answer = re.search(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        prediction
    )

    if hash_answer:
        predicted_answer = hash_answer.group(1)
    else:
        # Nếu model không sinh ####, lấy số cuối cùng
        numbers = re.findall(
            r"[-+]?\d+(?:\.\d+)?",
            prediction
        )

        predicted_answer = numbers[-1] if numbers else None

    correct = predicted_answer == gold_answer

    results_sft.append({
        "index": i + 1,
        "predicted": predicted_answer,
        "gold": gold_answer,
        "correct": correct,
        "latency": latency,
        "tokens": len(generated_tokens)
    })

    print(
        f"[{i+1}] "
        f"Predicted={predicted_answer} | "
        f"Gold={gold_answer} | "
        f"{correct} | "
        f"{latency:.2f}s | "
        f"{len(generated_tokens)} tokens"
    )

correct_count = sum(
    r["correct"] for r in results_sft
)

accuracy = (
    correct_count / len(results_sft) * 100
)

avg_latency = sum(
    r["latency"] for r in results_sft
) / len(results_sft)

avg_tokens = sum(
    r["tokens"] for r in results_sft
) / len(results_sft)

print("\n===== SFT-CoT BENCHMARK SUMMARY =====")
print(f"Correct: {correct_count}/10")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Average latency: {avg_latency:.2f}s")
print(f"Average generated tokens: {avg_tokens:.2f}")

===== SFT-CoT BENCHMARK — 10 SAMPLES =====
[1] Predicted=18 | Gold=18 | True | 8.38s | 156 tokens
[2] Predicted=3.0 | Gold=3 | False | 13.54s | 256 tokens
[3] Predicted=00 | Gold=70000 | False | 13.69s | 256 tokens
[4] Predicted=180 | Gold=540 | False | 8.48s | 162 tokens
[5] Predicted=20 | Gold=20 | True | 13.61s | 256 tokens
[6] Predicted=64 | Gold=64 | True | 7.62s | 143 tokens
[7] Predicted=260 | Gold=260 | True | 8.95s | 123 tokens
[8] Predicted=120 | Gold=160 | False | 11.81s | 222 tokens
[9] Predicted=165 | Gold=45 | False | 13.56s | 256 tokens
[10] Predicted=400 | Gold=460 | False | 13.70s | 256 tokens

===== SFT-CoT BENCHMARK SUMMARY =====
Correct: 4/10
Accuracy: 40.00%
Average latency: 11.34s
Average generated tokens: 208.60


# USC-DPO — REASONING CANDIDATE GENERATION

In [ ]:

# USC-DPO: Prepare a larger training pool

NUM_QUESTIONS = 80
NUM_CANDIDATES_PER_QUESTION = 4

candidate_data = []

for i in range(NUM_QUESTIONS):
    example = gsm8k["train"][i]

    candidate_data.append({
        "question_index": i,
        "question": example["question"],
        "gold_answer": example["answer"],
    })

print("===== USC-DPO TRAINING DATA =====")
print(f"Questions: {len(candidate_data)}")
print(f"Candidates per question: {NUM_CANDIDATES_PER_QUESTION}")
print(f"Target candidate count: {len(candidate_data) * NUM_CANDIDATES_PER_QUESTION}")
print("\nSample question:")
print(candidate_data[0]["question"])
print("\nGold solution:")
print(candidate_data[0]["gold_answer"])


===== USC-DPO TRAINING DATA =====
Questions: 80
Candidates per question: 4
Target candidate count: 320

Sample question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Gold solution:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [ ]:

# USC-DPO — Optimized Candidate Generation
# Replaces the original Cell 24.
# Compatible with Cell 23: 80 questions x 4 candidates.

import os
import gc
import json
import time
import torch
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
NUM_CANDIDATES = NUM_CANDIDATES_PER_QUESTION
MAX_NEW_TOKENS = 384
TEMPERATURE = 0.85
TOP_P = 0.95
SEED = 42

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
SFT_ADAPTER_DIR = os.path.join(ARTIFACT_DIR, "sft_lora_adapter")
PROGRESS_PATH = os.path.join(
    ARTIFACT_DIR, "candidate_generation_80x4_progress.json"
)
RAW_PATH = os.path.join(
    ARTIFACT_DIR, "candidate_generation_raw.json"
)

os.makedirs(ARTIFACT_DIR, exist_ok=True)

if "candidate_data" not in globals() or not candidate_data:
    raise RuntimeError("Run Cell 23 first.")

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Colab before running Cell 24.")

# 1. Preserve SFT adapter before releasing its model from memory.
if "trainer" in globals() and getattr(trainer, "model", None) is not None:
    trainer.model.save_pretrained(SFT_ADAPTER_DIR)
    tokenizer.save_pretrained(SFT_ADAPTER_DIR)
    print("Saved SFT adapter:", SFT_ADAPTER_DIR)

if not os.path.isfile(
    os.path.join(SFT_ADAPTER_DIR, "adapter_config.json")
):
    raise RuntimeError(
        "SFT adapter not safely saved. Stop here; do not release the SFT model."
    )

# Remove all known references to the SFT model before loading the generator.
for name in ("sft_model", "trainer", "model"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# 2. Load the base model for candidate generation.
if "tokenizer" not in globals():
    raise RuntimeError("Tokenizer missing; rerun the setup cells first.")

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if "candidate_model" not in globals():
    candidate_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

candidate_model.eval()

# 3. Resume completed candidates if this cell is interrupted.
multi_candidates = []

if os.path.isfile(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        saved = json.load(f)

    expected = len(candidate_data) * NUM_CANDIDATES

    # Reuse only data created for the same question pool and candidate count.
    if (
        saved.get("num_questions") == len(candidate_data)
        and saved.get("num_candidates") == NUM_CANDIDATES
        and saved.get("question_pool") == [
            x["question"] for x in candidate_data
        ]
    ):
        multi_candidates = saved.get("candidates", [])
        print("Resuming saved progress:", len(multi_candidates), "candidates")

def save_progress():
    payload = {
        "num_questions": len(candidate_data),
        "num_candidates": NUM_CANDIDATES,
        "question_pool": [x["question"] for x in candidate_data],
        "candidates": multi_candidates,
    }

    with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)

def generate_reasoning_candidates(question, count, seed):
    prompt = (
        "Solve this math word problem carefully. Show concise reasoning "
        "with the key calculations. Avoid repeated explanations and "
        "unnecessary self-talk. Check the arithmetic before answering.\n"
        "End with the final answer in this format: #### <number>\n\n"
        f"Problem:\n{question}\n\n"
        "Solution:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(candidate_model.device)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    with torch.inference_mode():
        outputs = candidate_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            num_return_sequences=count,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    start = inputs["input_ids"].shape[1]

    texts = [
        tokenizer.decode(
            output[start:],
            skip_special_tokens=True,
        ).strip()
        for output in outputs
    ]

    del inputs, outputs
    return texts

# 4. Generate four candidates per question.
start_time = time.time()

for position, item in enumerate(candidate_data, start=1):
    qid = item["question_index"]

    existing_ids = {
        c["candidate_id"]
        for c in multi_candidates
        if c["question_index"] == qid
    }

    missing_ids = [
        i for i in range(NUM_CANDIDATES)
        if i not in existing_ids
    ]

    if missing_ids:
        texts = generate_reasoning_candidates(
            item["question"],
            len(missing_ids),
            SEED + qid,
        )

        for candidate_id, solution in zip(missing_ids, texts):
            multi_candidates.append({
                "question_index": qid,
                "question": item["question"],
                "gold_answer": item["gold_answer"],
                "candidate_id": candidate_id,
                "generated_solution": solution,
            })

        # Save after each question so an interruption does not waste all work.
        save_progress()

    if position % 10 == 0 or position == len(candidate_data):
        print(
            f"Questions: {position}/{len(candidate_data)}"
            f" | Candidates saved: {len(multi_candidates)}"
            f" | Elapsed: {(time.time() - start_time) / 60:.1f} min",
            flush=True,
        )

# 5. Sort and save the complete dataset for downstream cells.
multi_candidates.sort(
    key=lambda c: (c["question_index"], c["candidate_id"])
)

with open(RAW_PATH, "w", encoding="utf-8") as f:
    json.dump(multi_candidates, f, ensure_ascii=False, indent=2)

print("\n===== GENERATION COMPLETE =====")
print("Questions:", len(candidate_data))
print("Candidates:", len(multi_candidates))
print("Expected:", len(candidate_data) * NUM_CANDIDATES)
print(f"Elapsed: {(time.time() - start_time) / 60:.1f} minutes")
print("Saved:", RAW_PATH)
print("SFT adapter preserved:", SFT_ADAPTER_DIR)
print("\n===== SAMPLE CANDIDATE =====")
print(multi_candidates[0]["generated_solution"])


Saved SFT adapter: /content/USC_DPO_artifacts/sft_lora_adapter


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Questions: 10/80 | Candidates saved: 40 | Elapsed: 2.0 min
Questions: 20/80 | Candidates saved: 80 | Elapsed: 3.7 min
Questions: 30/80 | Candidates saved: 120 | Elapsed: 5.5 min
Questions: 40/80 | Candidates saved: 160 | Elapsed: 7.1 min
Questions: 50/80 | Candidates saved: 200 | Elapsed: 8.6 min
Questions: 60/80 | Candidates saved: 240 | Elapsed: 10.4 min
Questions: 70/80 | Candidates saved: 280 | Elapsed: 12.4 min
Questions: 80/80 | Candidates saved: 320 | Elapsed: 14.2 min

===== GENERATION COMPLETE =====
Questions: 80
Candidates: 320
Expected: 320
Elapsed: 14.2 minutes
Saved: /content/USC_DPO_artifacts/candidate_generation_raw.json
SFT adapter preserved: /content/USC_DPO_artifacts/sft_lora_adapter

===== SAMPLE CANDIDATE =====
In April, Natalia sold 48 clips.
In May, she sold half as many clips as in April, which is \( \frac{48}{2} = 24 \) clips.
To find the total number of clips sold in April and May, we add the clips sold in each month: \( 48 + 24 = 72 \).
Therefore, Natalia sold

In [ ]:
# USC-DPO: Verify all generated candidates

import re

def extract_final_number(text):
    # 1. Ưu tiên đáp án trong \boxed{...}
    boxed = re.findall(
        r"\\boxed\{\s*([-+]?\d+(?:\.\d+)?)\s*\}",
        text
    )
    if boxed:
        return boxed[-1]

    # 2. Tìm đáp án sau ####
    match = re.search(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        text
    )
    if match:
        return match.group(1)

    # 3. Lấy số cuối cùng xuất hiện ở dòng cuối có chứa số
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    for line in reversed(lines):
        numbers = re.findall(
            r"[-+]?\d+(?:\.\d+)?",
            line
        )
        if numbers:
            return numbers[-1]

    return None


def normalize_number(value):
    if value is None:
        return None

    try:
        number = float(value)

        # 72.0 -> 72
        if number.is_integer():
            return str(int(number))

        return str(number)
    except:
        return str(value).strip()


def verify_candidate(candidate):
    predicted = extract_final_number(
        candidate["generated_solution"]
    )

    gold = extract_final_number(
        candidate["gold_answer"]
    )

    predicted_normalized = normalize_number(predicted)
    gold_normalized = normalize_number(gold)

    candidate["predicted_answer"] = predicted_normalized
    candidate["gold_final_answer"] = gold_normalized
    candidate["is_correct"] = (
        predicted_normalized is not None
        and gold_normalized is not None
        and predicted_normalized == gold_normalized
    )

    return candidate


# Verify toàn bộ 30 candidates
verified_candidates = []

for candidate in multi_candidates:
    verified_candidate = verify_candidate(candidate.copy())
    verified_candidates.append(verified_candidate)


# Thống kê
num_correct = sum(
    candidate["is_correct"]
    for candidate in verified_candidates
)

num_total = len(verified_candidates)

print(f"Total candidates: {num_total}")
print(f"Correct candidates: {num_correct}")
print(f"Candidate accuracy: {num_correct / num_total:.2%}")


# Hiển thị kết quả theo từng câu hỏi
for question_index in range(NUM_QUESTIONS):
    group = [
        candidate
        for candidate in verified_candidates
        if candidate["question_index"] == question_index
    ]

    print(f"\n===== QUESTION {question_index} =====")

    for candidate in group:
        print(
            f"Candidate {candidate['candidate_id']} | "
            f"Predicted: {candidate['predicted_answer']} | "
            f"Gold: {candidate['gold_final_answer']} | "
            f"Correct: {candidate['is_correct']}"
        )

Total candidates: 320
Correct candidates: 246
Candidate accuracy: 76.88%

===== QUESTION 0 =====
Candidate 0 | Predicted: 72 | Gold: 72 | Correct: True
Candidate 1 | Predicted: 72 | Gold: 72 | Correct: True
Candidate 2 | Predicted: 72 | Gold: 72 | Correct: True
Candidate 3 | Predicted: 40 | Gold: 72 | Correct: False

===== QUESTION 1 =====
Candidate 0 | Predicted: 10 | Gold: 10 | Correct: True
Candidate 1 | Predicted: 10 | Gold: 10 | Correct: True
Candidate 2 | Predicted: 10 | Gold: 10 | Correct: True
Candidate 3 | Predicted: 10 | Gold: 10 | Correct: True

===== QUESTION 2 =====
Candidate 0 | Predicted: 5 | Gold: 5 | Correct: True
Candidate 1 | Predicted: 45 | Gold: 5 | Correct: False
Candidate 2 | Predicted: 5 | Gold: 5 | Correct: True
Candidate 3 | Predicted: 52.5 | Gold: 5 | Correct: False

===== QUESTION 3 =====
Candidate 0 | Predicted: 42 | Gold: 42 | Correct: True
Candidate 1 | Predicted: 42 | Gold: 42 | Correct: True
Candidate 2 | Predicted: 42 | Gold: 42 | Correct: True
Candida

# USC-DPO — STEP FILTERING

In [ ]:
# USC-DPO: Step Segmentation

import re


def split_reasoning_steps(reasoning):
    """
    Tách reasoning thành các step.

    Trường hợp 1:
    Có đánh số rõ ràng:
        1. ...
        2. ...
        Step 1: ...
        Step 2: ...

    Trường hợp 2:
    Không có đánh số:
    Mỗi dòng reasoning không rỗng được xem là một step.
    """

    raw_lines = reasoning.splitlines()

    # Giữ lại các dòng không rỗng
    lines = [line.strip() for line in raw_lines if line.strip()]

    if not lines:
        return []

    # Kiểm tra xem reasoning có đánh số step hay không
    has_numbered_steps = any(
        re.match(
            r"^(?:\d+\.\s+|Step\s+\d+\s*:\s*)",
            line,
            re.IGNORECASE
        )
        for line in lines
    )

    # ---------------------------------------------------------
    # CASE 1: Có đánh số Step
    # ---------------------------------------------------------
    if has_numbered_steps:

        steps = []
        current_step = []

        for line in lines:

            is_new_step = re.match(
                r"^(?:\d+\.\s+|Step\s+\d+\s*:\s*)",
                line,
                re.IGNORECASE
            )

            if is_new_step:

                if current_step:
                    steps.append(" ".join(current_step))

                current_step = [line]

            else:

                if current_step:
                    current_step.append(line)
                else:
                    current_step = [line]

        if current_step:
            steps.append(" ".join(current_step))

        return steps

    # ---------------------------------------------------------
    # CASE 2: Không đánh số Step
    # Mỗi dòng reasoning là một step
    # ---------------------------------------------------------
    return lines


# Test trên Candidate 0
sample_candidate = verified_candidates[0]

sample_steps = split_reasoning_steps(
    sample_candidate["generated_solution"]
)

print("===== STEP SEGMENTATION =====")

for i, step in enumerate(sample_steps, start=1):
    print(f"\nStep {i}:")
    print(step)

print(f"\nTotal detected steps: {len(sample_steps)}")

===== STEP SEGMENTATION =====

Step 1:
In April, Natalia sold 48 clips.

Step 2:
In May, she sold half as many clips as in April, which is \( \frac{48}{2} = 24 \) clips.

Step 3:
To find the total number of clips sold in April and May, we add the clips sold in each month: \( 48 + 24 = 72 \).

Step 4:
Therefore, Natalia sold a total of 72 clips in April and May.

Step 5:
The answer is 72.

Total detected steps: 5


In [ ]:
# USC-DPO: Redundant Step Filtering

def normalize_step_text(text):
    """
    Chuẩn hóa text để phát hiện các step gần như trùng nhau.
    """
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\+\-\*/=\.]", "", text)
    return text


def is_redundant_step(step, previous_steps):
    """
    Phát hiện step thừa dựa trên:
    1. Step chỉ mang tính dẫn nhập, không chứa phép tính/thông tin mới.
    2. Step lặp lại gần như hoàn toàn một step trước đó.
    """

    text = step.lower().strip()

    # ---------------------------------------------------------
    # 1. Step dẫn nhập thuần túy
    # ---------------------------------------------------------
    introductory_patterns = [
        r"^to solve this problem",
        r"^to determine",
        r"^let'?s break",
        r"^let'?s calculate",
        r"^first,? let'?s",
        r"^we need to",
        r"^we are asked to",
        r"^in order to solve",
    ]

    has_number = bool(re.search(r"\d", text))
    has_operation = bool(
        re.search(r"[\+\-\*/=]", text)
    )

    for pattern in introductory_patterns:

        if re.search(pattern, text):

            # Chỉ loại nếu câu thực sự không chứa
            # thông tin/toán tử mới.
            if not has_number and not has_operation:
                return True

    # ---------------------------------------------------------
    # 2. Phát hiện step bị lặp lại
    # ---------------------------------------------------------
    normalized_current = normalize_step_text(step)

    for previous in previous_steps:

        normalized_previous = normalize_step_text(previous)

        if normalized_current == normalized_previous:
            return True

    return False


def filter_reasoning_steps(steps):
    """
    Giữ lại các step có giá trị suy luận,
    loại bỏ step dẫn nhập thuần túy hoặc bị lặp.
    """

    filtered_steps = []
    removed_steps = []

    for step in steps:

        if is_redundant_step(step, filtered_steps):
            removed_steps.append(step)
        else:
            filtered_steps.append(step)

    return filtered_steps, removed_steps


# Test trên Candidate 0
sample_steps = split_reasoning_steps(
    verified_candidates[0]["generated_solution"]
)

filtered_steps, removed_steps = filter_reasoning_steps(
    sample_steps
)

print("===== ORIGINAL STEPS =====")

for i, step in enumerate(sample_steps, start=1):
    print(f"\nStep {i}:")
    print(step)


print("\n===== FILTERED STEPS =====")

for i, step in enumerate(filtered_steps, start=1):
    print(f"\nStep {i}:")
    print(step)


print(f"\nOriginal step count: {len(sample_steps)}")
print(f"Filtered step count: {len(filtered_steps)}")
print(f"Removed step count: {len(removed_steps)}")

===== ORIGINAL STEPS =====

Step 1:
In April, Natalia sold 48 clips.

Step 2:
In May, she sold half as many clips as in April, which is \( \frac{48}{2} = 24 \) clips.

Step 3:
To find the total number of clips sold in April and May, we add the clips sold in each month: \( 48 + 24 = 72 \).

Step 4:
Therefore, Natalia sold a total of 72 clips in April and May.

Step 5:
The answer is 72.

===== FILTERED STEPS =====

Step 1:
In April, Natalia sold 48 clips.

Step 2:
In May, she sold half as many clips as in April, which is \( \frac{48}{2} = 24 \) clips.

Step 3:
To find the total number of clips sold in April and May, we add the clips sold in each month: \( 48 + 24 = 72 \).

Step 4:
Therefore, Natalia sold a total of 72 clips in April and May.

Step 5:
The answer is 72.

Original step count: 5
Filtered step count: 5
Removed step count: 0


In [ ]:
# USC-DPO: Apply Step Filtering to All Candidates

filtered_candidates = []

total_original_steps = 0
total_filtered_steps = 0
total_removed_steps = 0

for candidate in verified_candidates:

    # 1. Tách reasoning thành các step
    steps = split_reasoning_steps(
        candidate["generated_solution"]
    )

    # 2. Lọc step thừa / lặp
    filtered_steps, removed_steps = filter_reasoning_steps(
        steps
    )

    # 3. Lưu lại toàn bộ thông tin
    result = candidate.copy()

    result["steps"] = steps
    result["filtered_steps"] = filtered_steps
    result["removed_steps"] = removed_steps

    result["original_step_count"] = len(steps)
    result["filtered_step_count"] = len(filtered_steps)
    result["removed_step_count"] = len(removed_steps)

    filtered_candidates.append(result)

    total_original_steps += len(steps)
    total_filtered_steps += len(filtered_steps)
    total_removed_steps += len(removed_steps)


# ---------------------------------------------------------
# Tổng kết
# ---------------------------------------------------------

print("===== STEP FILTERING SUMMARY =====")

print(f"Total candidates: {len(filtered_candidates)}")
print(f"Original steps: {total_original_steps}")
print(f"Filtered steps: {total_filtered_steps}")
print(f"Removed steps: {total_removed_steps}")

if total_original_steps > 0:
    reduction_rate = (
        total_removed_steps / total_original_steps
    )

    print(f"Step reduction rate: {reduction_rate:.2%}")


# ---------------------------------------------------------
# Chi tiết từng candidate
# ---------------------------------------------------------

print("\n===== CANDIDATE RESULTS =====")

for candidate in filtered_candidates:

    print(
        f"Q{candidate['question_index']} | "
        f"C{candidate['candidate_id']} | "
        f"Correct: {candidate['is_correct']} | "
        f"Steps: {candidate['original_step_count']} -> "
        f"{candidate['filtered_step_count']} | "
        f"Removed: {candidate['removed_step_count']}"
    )

===== STEP FILTERING SUMMARY =====
Total candidates: 320
Original steps: 2199
Filtered steps: 2093
Removed steps: 106
Step reduction rate: 4.82%

===== CANDIDATE RESULTS =====
Q0 | C0 | Correct: True | Steps: 5 -> 5 | Removed: 0
Q0 | C1 | Correct: True | Steps: 4 -> 4 | Removed: 0
Q0 | C2 | Correct: True | Steps: 2 -> 2 | Removed: 0
Q0 | C3 | Correct: False | Steps: 15 -> 15 | Removed: 0
Q1 | C0 | Correct: True | Steps: 5 -> 5 | Removed: 0
Q1 | C1 | Correct: True | Steps: 7 -> 7 | Removed: 0
Q1 | C2 | Correct: True | Steps: 15 -> 15 | Removed: 0
Q1 | C3 | Correct: True | Steps: 4 -> 4 | Removed: 0
Q2 | C0 | Correct: True | Steps: 12 -> 12 | Removed: 0
Q2 | C1 | Correct: False | Steps: 9 -> 8 | Removed: 1
Q2 | C2 | Correct: True | Steps: 5 -> 5 | Removed: 0
Q2 | C3 | Correct: False | Steps: 2 -> 2 | Removed: 0
Q3 | C0 | Correct: True | Steps: 4 -> 4 | Removed: 0
Q3 | C1 | Correct: True | Steps: 5 -> 5 | Removed: 0
Q3 | C2 | Correct: True | Steps: 4 -> 4 | Removed: 0
Q3 | C3 | Correct: T

In [ ]:
# USC-DPO: Semantic Verification of Reasoning Steps

def parse_step_label(text):
    """
    Trích nhãn VALID / INVALID / REDUNDANT từ output verifier.
    """

    text_upper = text.upper().strip()

    # Ưu tiên các dạng kết luận rõ ràng
    if re.search(r"\bREDUNDANT\b", text_upper):
        return "REDUNDANT"

    if re.search(r"\bINVALID\b", text_upper):
        return "INVALID"

    if re.search(r"\bVALID\b", text_upper):
        return "VALID"

    # Một số cách diễn đạt tự nhiên của model
    if re.search(
        r"\b(MATHEMATICALLY|LOGICALLY)\s+CORRECT\b",
        text_upper
    ):
        return "VALID"

    if re.search(
        r"\b(CORRECT|VALID)\b.*\b(STEP|REASONING)\b",
        text_upper
    ):
        return "VALID"

    if re.search(
        r"\b(STEP|REASONING)\b.*\b(CORRECT|VALID)\b",
        text_upper
    ):
        return "VALID"

    if re.search(
        r"\b(ERROR|INCORRECT|WRONG)\b",
        text_upper
    ):
        return "INVALID"

    return "UNKNOWN"


def verify_reasoning_steps(candidate):
    """
    Kiểm tra từng step bằng cùng một SLM.

    Nhãn:
    - VALID
    - INVALID
    - REDUNDANT
    """

    steps = candidate["filtered_steps"]

    step_verification = []

    for i, step in enumerate(steps):

        # Chỉ đưa tối đa vài step trước đó vào context
        # để prompt không quá dài.
        previous_steps = steps[:i]

        if previous_steps:
            previous_context = "\n".join(
                f"Step {j + 1}: {previous_step}"
                for j, previous_step in enumerate(previous_steps[-3:])
            )
        else:
            previous_context = "(No previous steps.)"

        prompt = (
            "You are a strict mathematical reasoning verifier.\n\n"
            f"Problem:\n{candidate['question']}\n\n"
            f"Previous steps:\n{previous_context}\n\n"
            f"Current step:\n{step}\n\n"
            "Judge ONLY the current step.\n\n"
            "VALID: mathematically and logically correct.\n"
            "INVALID: contains a mathematical or logical error.\n"
            "REDUNDANT: repeats information without adding reasoning.\n\n"
            "Your final verdict must be exactly one of:\n"
            "VALID\n"
            "INVALID\n"
            "REDUNDANT\n\n"
            "Verdict:"
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(candidate_model.device)

        with torch.no_grad():
            outputs = candidate_model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_length = inputs["input_ids"].shape[1]

        verification_text = tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens=True
        ).strip()

        label = parse_step_label(
            verification_text
        )

        step_verification.append({
            "step": i + 1,
            "text": step,
            "label": label,
            "raw_output": verification_text,
        })

    candidate["step_verification"] = step_verification

    candidate["valid_step_count"] = sum(
        item["label"] == "VALID"
        for item in step_verification
    )

    candidate["invalid_step_count"] = sum(
        item["label"] == "INVALID"
        for item in step_verification
    )

    candidate["redundant_step_count"] = sum(
        item["label"] == "REDUNDANT"
        for item in step_verification
    )

    candidate["unknown_step_count"] = sum(
        item["label"] == "UNKNOWN"
        for item in step_verification
    )

    return candidate


# ---------------------------------------------------------
# Test trên Candidate 0
# ---------------------------------------------------------

test_candidate = filtered_candidates[0].copy()

test_candidate = verify_reasoning_steps(
    test_candidate
)

print("===== SEMANTIC STEP VERIFICATION =====")

for item in test_candidate["step_verification"]:
    print(
        f"Step {item['step']} | "
        f"{item['label']} | "
        f"{item['text']}"
    )

print("\n===== RAW VERIFIER OUTPUT =====")

for item in test_candidate["step_verification"]:
    print(
        f"Step {item['step']}: "
        f"{item['raw_output']}"
    )

print("\n===== SUMMARY =====")
print(f"Valid: {test_candidate['valid_step_count']}")
print(f"Invalid: {test_candidate['invalid_step_count']}")
print(f"Redundant: {test_candidate['redundant_step_count']}")
print(f"Unknown: {test_candidate['unknown_step_count']}")

===== SEMANTIC STEP VERIFICATION =====
Step 1 | VALID | In April, Natalia sold 48 clips.
Step 2 | VALID | In May, she sold half as many clips as in April, which is \( \frac{48}{2} = 24 \) clips.
Step 3 | VALID | To find the total number of clips sold in April and May, we add the clips sold in each month: \( 48 + 24 = 72 \).
Step 4 | VALID | Therefore, Natalia sold a total of 72 clips in April and May.
Step 5 | VALID | The answer is 72.

===== RAW VERIFIER OUTPUT =====
Step 1: VALID

Explanation:
To determine the total number of clips Natalia sold in April and May, we need to follow these steps:

1. Identify the number of
Step 2: VALID
To determine the total number of clips Natalia sold in April and May, we need to follow these steps:

1. Identify the number of clips sold
Step 3: VALID

To determine the total number of clips Natalia sold in April and May, we need to follow these steps:

1. Identify the number of clips sold
Step 4: VALID

Explanation:
The problem involves basic arithmeti

In [ ]:

# USC-DPO: Fast Pass-Through — Skip Expensive Semantic Verification
# Preserve downstream variable names without pretending steps were verified.

if "filtered_candidates" not in globals():
    raise RuntimeError(
        "filtered_candidates is missing. Do not restart runtime; "
        "check whether the earlier filtering cell completed."
    )

semantic_verified_candidates = []

total_valid_steps = 0
total_invalid_steps = 0
total_redundant_steps = 0
total_unknown_steps = 0

for candidate in filtered_candidates:
    result = candidate.copy()

    steps = result.get("filtered_steps", [])

    # These steps were not semantically evaluated.
    # Mark them UNKNOWN instead of falsely claiming they are valid.
    result["step_verification"] = [
        {
            "step": i + 1,
            "text": step,
            "label": "UNKNOWN",
            "raw_output": "Semantic verification skipped to avoid expensive per-step generation."
        }
        for i, step in enumerate(steps)
    ]

    result["valid_step_count"] = 0
    result["invalid_step_count"] = 0
    result["redundant_step_count"] = 0
    result["unknown_step_count"] = len(steps)
    result["semantic_verification_skipped"] = True

    semantic_verified_candidates.append(result)

    total_unknown_steps += len(steps)

print("===== USC-DPO — FAST VERIFICATION BYPASS =====")
print("Candidates preserved:", len(semantic_verified_candidates))
print("LLM generation calls: 0")
print("Valid steps claimed:", total_valid_steps)
print("Invalid steps claimed:", total_invalid_steps)
print("Steps left unverified:", total_unknown_steps)
print("Semantic verification: SKIPPED")
print("Ready for downstream processing.")


===== USC-DPO — FAST VERIFICATION BYPASS =====
Candidates preserved: 320
LLM generation calls: 0
Valid steps claimed: 0
Invalid steps claimed: 0
Steps left unverified: 2093
Semantic verification: SKIPPED
Ready for downstream processing.


In [ ]:
# USC-DPO: Candidate Quality Summary

for candidate in semantic_verified_candidates:

    # Outcome correctness vẫn được xác định bằng gold answer
    outcome_correct = candidate["is_correct"]

    # Reasoning quality dựa trên semantic verification
    has_invalid_step = (
        candidate["invalid_step_count"] > 0
    )

    has_unknown_step = (
        candidate["unknown_step_count"] > 0
    )

    # Candidate đạt chất lượng đầy đủ khi:
    # - đáp án đúng
    # - không có step INVALID
    # - không có step UNKNOWN
    candidate["is_high_quality"] = (
        outcome_correct
        and not has_invalid_step
        and not has_unknown_step
    )

    # Candidate lỗi rõ ràng khi:
    # - đáp án sai hoặc
    # - có step INVALID
    candidate["is_low_quality"] = (
        not outcome_correct
        or has_invalid_step
    )


# ---------------------------------------------------------
# Tổng kết
# ---------------------------------------------------------

high_quality_count = sum(
    candidate["is_high_quality"]
    for candidate in semantic_verified_candidates
)

low_quality_count = sum(
    candidate["is_low_quality"]
    for candidate in semantic_verified_candidates
)

print("===== CANDIDATE QUALITY SUMMARY =====")

print(
    f"Total candidates: "
    f"{len(semantic_verified_candidates)}"
)

print(
    f"High-quality candidates: "
    f"{high_quality_count}"
)

print(
    f"Low-quality candidates: "
    f"{low_quality_count}"
)


# ---------------------------------------------------------
# Chi tiết
# ---------------------------------------------------------

print("\n===== QUALITY BY CANDIDATE =====")

for candidate in semantic_verified_candidates:

    print(
        f"Q{candidate['question_index']} | "
        f"C{candidate['candidate_id']} | "
        f"Correct: {candidate['is_correct']} | "
        f"Invalid steps: {candidate['invalid_step_count']} | "
        f"Unknown: {candidate['unknown_step_count']} | "
        f"Steps: {candidate['filtered_step_count']} | "
        f"High-quality: {candidate['is_high_quality']}"
    )

===== CANDIDATE QUALITY SUMMARY =====
Total candidates: 320
High-quality candidates: 0
Low-quality candidates: 74

===== QUALITY BY CANDIDATE =====
Q0 | C0 | Correct: True | Invalid steps: 0 | Unknown: 5 | Steps: 5 | High-quality: False
Q0 | C1 | Correct: True | Invalid steps: 0 | Unknown: 4 | Steps: 4 | High-quality: False
Q0 | C2 | Correct: True | Invalid steps: 0 | Unknown: 2 | Steps: 2 | High-quality: False
Q0 | C3 | Correct: False | Invalid steps: 0 | Unknown: 15 | Steps: 15 | High-quality: False
Q1 | C0 | Correct: True | Invalid steps: 0 | Unknown: 5 | Steps: 5 | High-quality: False
Q1 | C1 | Correct: True | Invalid steps: 0 | Unknown: 7 | Steps: 7 | High-quality: False
Q1 | C2 | Correct: True | Invalid steps: 0 | Unknown: 15 | Steps: 15 | High-quality: False
Q1 | C3 | Correct: True | Invalid steps: 0 | Unknown: 4 | Steps: 4 | High-quality: False
Q2 | C0 | Correct: True | Invalid steps: 0 | Unknown: 12 | Steps: 12 | High-quality: False
Q2 | C1 | Correct: False | Invalid steps: 0 

# USC-DPO — SELF-CORRECTION

In [ ]:

# USC-DPO — Corrected Mixed Preference Dataset
# Reuses existing candidate_data and multi_candidates.
# No model generation and no GPU training in this cell.

import os
import re
import json
import zipfile
from decimal import Decimal

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

if not globals().get("multi_candidates"):
    raise RuntimeError(
        "multi_candidates is missing. Do not regenerate candidates blindly."
    )

if not globals().get("candidate_data"):
    raise RuntimeError("candidate_data is missing.")

# =========================================================
# 1. Number normalization and result parsing
# =========================================================

def usc34_normalize_number(value):
    try:
        value = str(value).strip().replace(",", "").replace("$", "")
        number = Decimal(value)
        if not number.is_finite():
            return None
        if number == number.to_integral_value():
            return str(int(number))
        return format(number.normalize(), "f")
    except Exception:
        return None


def usc34_last_number(text):
    """Return the last simple numeric literal in text."""
    values = re.findall(
        r"[-+]?\d[\d,]*(?:\.\d+)?",
        str(text or "")
    )
    if not values:
        return None
    return usc34_normalize_number(values[-1])


def usc34_result_after_equals(text):
    """
    If an answer expression contains '=', use the result after the
    LAST equals sign, e.g. 12 * (50/60) = 10 -> 10.
    """
    text = str(text or "")
    if "=" in text:
        text = text.rsplit("=", 1)[1]
    return usc34_last_number(text)


def usc34_answer(text):
    text = str(text or "")
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if not lines:
        return None

    # Priority 1: explicit GSM8K #### final-answer line.
    for line in reversed(lines):
        if "####" in line:
            suffix = line.split("####", 1)[1].strip()
            if suffix:
                result = usc34_result_after_equals(suffix)
                if result is not None:
                    return result

    # Priority 2: simple numeric boxed answers.
    # Intentionally only parses a box containing a plain number.
    boxed = re.compile(
        r"\\boxed\s*\{\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*\}"
    )
    boxed_matches = list(boxed.finditer(text))
    if boxed_matches:
        result = usc34_normalize_number(boxed_matches[-1].group(1))
        if result is not None:
            return result

    # Priority 3: answer cue lines. Parse the RESULT of an equation,
    # not the first number appearing in an expression.
    answer_cue = re.compile(
        r"\b(?:final answer|the answer is|answer is|answer)\b\s*:?\s*",
        re.IGNORECASE
    )

    for line in reversed(lines):
        match = answer_cue.search(line)
        if match:
            suffix = line[match.end():].strip()
            result = usc34_result_after_equals(suffix)
            if result is not None:
                return result

    # Priority 4: final line fallback.
    # Use the result after the last '=' where available.
    return usc34_result_after_equals(lines[-1])


# Parser regression checks: do not proceed if any fails.
_parser_tests = [
    ("Work\n#### 5", "5"),
    ("Final answer: $1,250", "1250"),
    (r"Answer: \boxed{42}", "42"),
    ("The answer is 12 * (50/60) = 10", "10"),
    ("####\nThe answer is 12 * (50/60) = 10", "10"),
    ("Calculation: 8 + 7 = 15\n#### 15", "15"),
]

for test_text, expected in _parser_tests:
    actual = usc34_answer(test_text)
    assert actual == expected, (
        f"Parser regression failed: expected {expected}, got {actual}; "
        f"text={test_text!r}"
    )

print("Answer-parser regression checks: PASS")


# =========================================================
# 2. Conservative text filtering
# =========================================================

USC34_INTRO_PATTERNS = [
    r"^let'?s (?:solve|understand|break down|think through)",
    r"^let us (?:solve|understand|calculate)",
    r"^to solve this problem",
    r"^first,? let'?s",
    r"^we need to (?:find|calculate|determine|solve)",
    r"^we are asked to (?:find|calculate|determine)",
    r"^in order to solve",
    r"^i will (?:solve|calculate|work through)",
    r"^let me (?:think|calculate|solve)",
    r"^let'?s think step by step",
]


def usc34_clean(text):
    text = str(text or "")
    text = re.sub(r"<<.*?>>", "", text)
    text = re.sub(r"\n[ \t]*\n[ \t]*\n+", "\n\n", text)
    return "\n".join(line.rstrip() for line in text.splitlines()).strip()


def usc34_normalize_line(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def usc34_filter_steps(text):
    """
    Only remove:
      1. obvious introductory/meta lines without numbers or operators;
      2. consecutive duplicate non-mathematical lines.

    Never remove a line containing numeric content or a math operator.
    Without a semantic verifier, this cannot prove a step is redundant.
    """
    original = usc34_clean(text)
    lines = [line.strip() for line in original.splitlines() if line.strip()]

    kept = []
    removed = []
    previous_kept = None

    for line in lines:
        normalized = usc34_normalize_line(line)

        is_answer_line = bool(re.search(
            r"####|\\boxed|final answer|the answer is|answer\s*:",
            line,
            flags=re.IGNORECASE
        ))

        # Protect numeric or mathematical content.
        has_math_content = bool(re.search(r"\d|[=+*/<>]", line))

        is_intro = (
            not is_answer_line
            and not has_math_content
            and any(
                re.search(pattern, line, flags=re.IGNORECASE)
                for pattern in USC34_INTRO_PATTERNS
            )
        )

        if is_intro:
            removed.append(line)
            continue

        # Only remove an adjacent duplicate sentence, never a math line.
        is_safe_duplicate = (
            not is_answer_line
            and not has_math_content
            and previous_kept is not None
            and normalized == usc34_normalize_line(previous_kept)
        )

        if is_safe_duplicate:
            removed.append(line)
            continue

        kept.append(line)
        previous_kept = line

    filtered = "\n".join(kept).strip() or original

    original_answer = usc34_answer(original)
    filtered_answer = usc34_answer(filtered)

    # Do not use a transformed completion if its parsed answer changes.
    if original_answer is None or filtered_answer != original_answer:
        return {
            "text": original,
            "removed_lines": [],
            "removed_characters": 0,
            "fallback": True,
        }

    return {
        "text": filtered,
        "removed_lines": removed,
        "removed_characters": max(0, len(original) - len(filtered)),
        "fallback": False,
    }


# =========================================================
# 3. Build gold-reference lookup
# =========================================================

gold_by_qid = {}

for item in candidate_data:
    qid = int(item["question_index"])
    question = str(item["question"]).strip()
    gold_raw = usc34_clean(item["gold_answer"])
    gold_answer = usc34_answer(gold_raw)

    if not question or not gold_raw or gold_answer is None:
        continue

    filtered_gold = usc34_filter_steps(gold_raw)
    gold_solution = filtered_gold["text"]

    if usc34_answer(gold_solution) != gold_answer:
        gold_solution = gold_raw

    gold_by_qid[qid] = {
        "question": question,
        "gold_answer": gold_answer,
        "gold_solution": gold_solution,
    }


def usc34_make_prompt(question):
    return (
        "Solve this math word problem carefully. Use concise reasoning "
        "and keep useful calculations. Avoid repeated explanations and "
        "unnecessary self-talk.\n"
        "End with exactly one final line in this format: #### <number>\n"
        "Do not write anything after the final-answer line.\n\n"
        f"Problem:\n{question}\n\nSolution:\n"
    )


# =========================================================
# 4. Answer-correction pairs
# =========================================================

correction_pairs = []
audit = []
seen_wrong_outputs = set()

correct_count = 0
incorrect_count = 0
unparsed_count = 0

for candidate in multi_candidates:
    qid = int(candidate["question_index"])
    cid = int(candidate["candidate_id"])

    if qid not in gold_by_qid:
        continue

    gold = gold_by_qid[qid]
    original = usc34_clean(candidate.get("generated_solution", ""))
    predicted = usc34_answer(original)

    if predicted is None:
        unparsed_count += 1
        audit.append({
            "question_index": qid,
            "candidate_id": cid,
            "status": "unparsed",
            "gold": gold["gold_answer"],
        })
        continue

    if predicted == gold["gold_answer"]:
        correct_count += 1
        audit.append({
            "question_index": qid,
            "candidate_id": cid,
            "status": "correct_candidate",
            "prediction": predicted,
            "gold": gold["gold_answer"],
        })
        continue

    incorrect_count += 1
    dedup_key = (
        qid,
        re.sub(r"\s+", " ", original.lower()).strip()
    )

    if dedup_key in seen_wrong_outputs:
        continue
    seen_wrong_outputs.add(dedup_key)

    chosen = gold["gold_solution"]

    pair = {
        "pair_type": "answer_correction",
        "prompt": usc34_make_prompt(gold["question"]),
        "chosen": chosen,
        "rejected": original,
        "question_index": qid,
        "question": gold["question"],
        "chosen_candidate_id": "gsm8k_reference_solution",
        "rejected_candidate_id": cid,
        "chosen_answer": gold["gold_answer"],
        "rejected_answer": predicted,
        "gold_answer": gold["gold_answer"],
        "chosen_is_correct": True,
        "rejected_is_correct": False,
    }

    assert usc34_answer(pair["chosen"]) == pair["gold_answer"]
    assert usc34_answer(pair["rejected"]) != pair["gold_answer"]

    correction_pairs.append(pair)
    audit.append({
        "question_index": qid,
        "candidate_id": cid,
        "status": "answer_correction_pair",
        "prediction": predicted,
        "gold": gold["gold_answer"],
    })


# =========================================================
# 5. Step-efficiency pairs
# =========================================================

# Use only correct original answers; both sides must retain the gold answer.
# The shorter side differs only by the conservative filter above.

efficiency_candidates = {}
MIN_REMOVED_CHARACTERS = 20
MIN_REDUCTION_RATIO = 0.05
MAX_STEP_EFFICIENCY_PAIRS = 20

for candidate in multi_candidates:
    qid = int(candidate["question_index"])
    cid = int(candidate["candidate_id"])

    if qid not in gold_by_qid:
        continue

    gold = gold_by_qid[qid]
    original = usc34_clean(candidate.get("generated_solution", ""))
    answer = usc34_answer(original)

    if answer is None or answer != gold["gold_answer"]:
        continue

    filtered = usc34_filter_steps(original)

    if filtered["fallback"] or not filtered["removed_lines"]:
        continue

    chosen = filtered["text"]
    if usc34_answer(chosen) != gold["gold_answer"]:
        continue

    removed_chars = filtered["removed_characters"]
    reduction_ratio = removed_chars / max(1, len(original))

    if removed_chars < MIN_REMOVED_CHARACTERS:
        continue
    if reduction_ratio < MIN_REDUCTION_RATIO:
        continue

    score = (len(filtered["removed_lines"]), removed_chars, reduction_ratio)
    existing = efficiency_candidates.get(qid)

    if existing is None or score > existing["score"]:
        efficiency_candidates[qid] = {
            "candidate": candidate,
            "original": original,
            "filtered": filtered,
            "score": score,
        }

ranked_efficiency = sorted(
    efficiency_candidates.items(),
    key=lambda item: item[1]["score"],
    reverse=True,
)[:MAX_STEP_EFFICIENCY_PAIRS]

efficiency_pairs = []

for qid, item in ranked_efficiency:
    candidate = item["candidate"]
    gold = gold_by_qid[qid]
    chosen = item["filtered"]["text"]
    rejected = item["original"]
    answer = usc34_answer(chosen)

    pair = {
        "pair_type": "step_efficiency",
        "prompt": usc34_make_prompt(gold["question"]),
        "chosen": chosen,
        "rejected": rejected,
        "question_index": qid,
        "question": gold["question"],
        "chosen_candidate_id": candidate["candidate_id"],
        "rejected_candidate_id": candidate["candidate_id"],
        "chosen_answer": answer,
        "rejected_answer": usc34_answer(rejected),
        "gold_answer": gold["gold_answer"],
        "chosen_is_correct": True,
        "rejected_is_correct": True,
        "removed_lines": item["filtered"]["removed_lines"],
        "removed_characters": item["filtered"]["removed_characters"],
    }

    assert usc34_answer(pair["chosen"]) == pair["gold_answer"]
    assert usc34_answer(pair["rejected"]) == pair["gold_answer"]
    assert len(pair["chosen"]) < len(pair["rejected"])

    efficiency_pairs.append(pair)
    audit.append({
        "question_index": qid,
        "candidate_id": candidate["candidate_id"],
        "status": "step_efficiency_pair",
        "gold": gold["gold_answer"],
        "removed_lines": pair["removed_lines"],
        "removed_characters": pair["removed_characters"],
    })


# =========================================================
# 6. Validate and save
# =========================================================

pairs = correction_pairs + efficiency_pairs

if not correction_pairs:
    raise RuntimeError("No answer-correction pairs. Stop before training.")

for i, pair in enumerate(pairs):
    if not pair["prompt"] or not pair["chosen"] or not pair["rejected"]:
        raise ValueError(f"Empty completion at pair {i}.")

    if pair["pair_type"] == "answer_correction":
        assert pair["chosen_answer"] == pair["gold_answer"]
        assert pair["rejected_answer"] != pair["gold_answer"]
    else:
        assert pair["chosen_answer"] == pair["gold_answer"]
        assert pair["rejected_answer"] == pair["gold_answer"]

# Cell 35 consumes this variable. It must also be updated to accept both types.
dpo_dataset_validated = pairs

pairs_path = os.path.join(
    ARTIFACT_DIR, "dpo_preference_pairs_original_validated.json"
)
audit_path = os.path.join(
    ARTIFACT_DIR, "usc_dpo_gold_reference_pairs_audit.json"
)
summary_path = os.path.join(
    ARTIFACT_DIR, "usc_dpo_gold_reference_summary.json"
)

with open(pairs_path, "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

summary = {
    "raw_candidates": len(multi_candidates),
    "gold_questions": len(gold_by_qid),
    "correct_candidates": correct_count,
    "incorrect_candidates": incorrect_count,
    "unparsed_candidates": unparsed_count,
    "answer_correction_pairs": len(correction_pairs),
    "step_efficiency_pairs": len(efficiency_pairs),
    "total_preference_pairs": len(pairs),
    "correction_questions": len({
        p["question_index"] for p in correction_pairs
    }),
    "efficiency_questions": len({
        p["question_index"] for p in efficiency_pairs
    }),
    "semantic_verifier_used": False,
    "step_filter_type": "conservative text heuristic",
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

zip_path = "/content/USC_DPO_original_validated_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in (pairs_path, audit_path, summary_path):
        zf.write(path, arcname=os.path.basename(path))

print("\n" + "=" * 64)
print("USC-DPO — CORRECTED MIXED PREFERENCE DATA SUMMARY")
print("=" * 64)
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nParser check example:")
example = "The answer is 12 * (50/60) = 10"
print("Text:", example)
print("Parsed answer:", usc34_answer(example))

print("\nSample answer-correction pair:")
print(json.dumps(
    correction_pairs[0],
    ensure_ascii=False,
    indent=2
)[:2500])

if efficiency_pairs:
    print("\nSample step-efficiency pair:")
    print(json.dumps(
        efficiency_pairs[0],
        ensure_ascii=False,
        indent=2
    )[:2500])

print("\nSaved pairs:", pairs_path)
print("Saved audit:", audit_path)
print("ZIP:", zip_path)


Answer-parser regression checks: PASS

USC-DPO — CORRECTED MIXED PREFERENCE DATA SUMMARY
raw_candidates: 320
gold_questions: 80
correct_candidates: 241
incorrect_candidates: 61
unparsed_candidates: 18
answer_correction_pairs: 61
step_efficiency_pairs: 20
total_preference_pairs: 81
correction_questions: 36
efficiency_questions: 20
semantic_verifier_used: False
step_filter_type: conservative text heuristic

Parser check example:
Text: The answer is 12 * (50/60) = 10
Parsed answer: 10

Sample answer-correction pair:
{
  "pair_type": "answer_correction",
  "prompt": "Solve this math word problem carefully. Use concise reasoning and keep useful calculations. Avoid repeated explanations and unnecessary self-talk.\nEnd with exactly one final line in this format: #### <number>\nDo not write anything after the final-answer line.\n\nProblem:\nBetty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, 

In [ ]:

# USC-DPO — Mixed Preference Retraining (Robust Label Validation)
# Supports:
#   1. answer_correction: chosen correct, rejected incorrect
#   2. step_efficiency: both correct, chosen shorter
#
# Reuses the existing 81 preference pairs.
# Stored labels from Cell 34 are authoritative; a second parser is
# used only as a cleanup safety check, not as the sole source of labels.

import os
import re
import gc
import json
import random
import zipfile
from decimal import Decimal

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model


# =========================================================
# 1. Configuration
# =========================================================

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"

NUM_EPOCHS = 3
LEARNING_RATE = 1e-6
DPO_BETA = 0.1

MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 384
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
SFT_ADAPTER_DIR = os.path.join(
    ARTIFACT_DIR, "sft_lora_adapter"
)
ADAPTER_DIR = os.path.join(
    ARTIFACT_DIR, "dpo_lora_adapter"
)

os.makedirs(ARTIFACT_DIR, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

torch.cuda.manual_seed_all(SEED)

if not os.path.isfile(
    os.path.join(SFT_ADAPTER_DIR, "adapter_config.json")
):
    raise FileNotFoundError(
        f"SFT adapter missing: {SFT_ADAPTER_DIR}"
    )


# =========================================================
# 2. Load the existing preference pairs
# =========================================================

training_pairs_raw = globals().get(
    "dpo_dataset_validated", None
)

if not training_pairs_raw:
    pairs_path = os.path.join(
        ARTIFACT_DIR,
        "dpo_preference_pairs_original_validated.json",
    )

    if not os.path.isfile(pairs_path):
        raise FileNotFoundError(
            "Preference dataset missing. No training started."
        )

    with open(pairs_path, "r", encoding="utf-8") as f:
        training_pairs_raw = json.load(f)

if not training_pairs_raw:
    raise RuntimeError("Preference dataset is empty.")


# =========================================================
# 3. Answer parsing and conservative completion cleanup
# =========================================================

def normalize_number(value):
    try:
        value = str(value).strip().replace(",", "").replace("$", "")
        number = Decimal(value)

        if not number.is_finite():
            return None

        if number == number.to_integral_value():
            return str(int(number))

        return format(number.normalize(), "f")
    except Exception:
        return None


def last_number(text):
    values = re.findall(
        r"[-+]?\d[\d,]*(?:\.\d+)?",
        str(text or ""),
    )
    if not values:
        return None
    return normalize_number(values[-1])


def parse_answer(text):
    """Best-effort independent parser. Stored labels remain authoritative."""
    text = str(text or "")
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if not lines:
        return None

    # GSM8K marker: use the result after the last '=' on the answer line.
    for line in reversed(lines):
        if "####" in line:
            suffix = line.split("####", 1)[1].strip()
            if suffix:
                result = last_number(suffix.rsplit("=", 1)[-1])
                if result is not None:
                    return result

    # Simple boxed number.
    boxed = list(re.finditer(
        r"\\boxed\s*\{\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*\}",
        text,
    ))
    if boxed:
        result = normalize_number(boxed[-1].group(1))
        if result is not None:
            return result

    # Explicit answer cue. Handles "12 * (50/60) = 10".
    answer_cue = re.compile(
        r"\b(?:final answer|the answer is|answer is|answer)\b\s*:?\s*",
        re.IGNORECASE,
    )

    for line in reversed(lines):
        match = answer_cue.search(line)
        if match:
            suffix = line[match.end():].strip()
            result = last_number(suffix.rsplit("=", 1)[-1])
            if result is not None:
                return result

    return last_number(lines[-1].rsplit("=", 1)[-1])


def balanced_box_end(text, box_start):
    brace_start = text.find("{", box_start)
    if brace_start < 0:
        return None

    depth = 0
    for pos in range(brace_start, len(text)):
        if text[pos] == "{":
            depth += 1
        elif text[pos] == "}":
            depth -= 1
            if depth == 0:
                return pos + 1

    return None


def trim_after_final_answer(text):
    """
    Remove fenced code blocks and trailing text after a clear final answer.
    Caller accepts the cleaned result only when its answer matches the
    dataset label; otherwise the original completion is preserved.
    """
    text = str(text or "").strip()

    text = re.sub(
        r"(?ms)^[ \t]*```[^\n]*\n.*?^[ \t]*```[ \t]*$",
        "",
        text,
    )
    text = re.sub(
        r"(?ms)^[ \t]*```[^\n]*\n.*\Z",
        "",
        text,
    )
    text = re.sub(r"<<.*?>>", "", text)
    text = "\n".join(
        line.rstrip() for line in text.splitlines()
    ).strip()

    lines = text.splitlines()

    # Prefer an explicit final-answer marker.
    for i in range(len(lines) - 1, -1, -1):
        if "####" not in lines[i]:
            continue

        suffix = lines[i].split("####", 1)[1].strip()
        if last_number(suffix.rsplit("=", 1)[-1]) is not None:
            return "\n".join(lines[:i + 1]).strip()

    # Next: preserve through the last boxed answer.
    box_starts = [
        match.start()
        for match in re.finditer(r"\\boxed\s*\{", text)
    ]

    if box_starts:
        end = balanced_box_end(text, box_starts[-1])
        if end is not None:
            return text[:end].strip()

    # Next: preserve through the last explicit answer-cue line.
    cue = re.compile(
        r"\b(?:final answer|the answer is|answer is|answer)\b\s*:?",
        re.IGNORECASE,
    )

    for i in range(len(lines) - 1, -1, -1):
        if cue.search(lines[i]):
            return "\n".join(lines[:i + 1]).strip()

    return text


# Parser regression checks.
assert parse_answer(
    "The answer is 12 * (50/60) = 10"
) == "10"

assert parse_answer("Work\n#### 16") == "16"

assert parse_answer(
    r"Answer: $\boxed{\color{blue}{45}}$"
) == "45"

print("Training answer-parser regression checks: PASS")


# =========================================================
# 4. Validate stored labels; clean only when safe
# =========================================================

training_pairs = []
validation_audit = []

dropped_efficiency_pairs = 0
cleanup_accepted = 0
cleanup_fallback = 0
parser_unparsed_after_cleanup = 0
parser_label_disagreements = 0

for i, raw in enumerate(training_pairs_raw):
    pair = dict(raw)

    pair_type = pair.get("pair_type", "answer_correction")

    if pair_type not in {"answer_correction", "step_efficiency"}:
        raise ValueError(
            f"Unknown pair_type at pair {i}: {pair_type!r}"
        )

    prompt = str(pair.get("prompt", "")).strip()
    original_chosen = str(pair.get("chosen", "")).strip()
    original_rejected = str(pair.get("rejected", "")).strip()

    if not prompt or not original_chosen or not original_rejected:
        raise ValueError(f"Empty prompt/completion at pair {i}.")

    # These labels were explicitly stored and validated in Cell 34.
    gold = normalize_number(pair.get("gold_answer"))
    chosen_label = normalize_number(pair.get("chosen_answer"))
    rejected_label = normalize_number(pair.get("rejected_answer"))

    if gold is None or chosen_label is None or rejected_label is None:
        raise ValueError(
            f"Missing stored answer labels at pair {i}. "
            "No GPU training started."
        )

    if chosen_label != gold:
        raise ValueError(
            f"Stored chosen label disagrees with gold at pair {i}. "
            "No GPU training started."
        )

    if pair_type == "answer_correction":
        if rejected_label == gold:
            raise ValueError(
                f"Correction pair {i} has a non-negative rejection label. "
                "No GPU training started."
            )
    else:
        if rejected_label != gold:
            raise ValueError(
                f"Efficiency pair {i} does not have two correct labels. "
                "No GPU training started."
            )

    # Try to clean trailing code/prose.
    candidate_chosen = trim_after_final_answer(original_chosen)
    candidate_rejected = trim_after_final_answer(original_rejected)

    parsed_chosen = parse_answer(candidate_chosen)
    parsed_rejected = parse_answer(candidate_rejected)

    # Accept a cleaned string only if the independent parser confirms its
    # answer matches the already-validated stored label.
    if parsed_chosen == chosen_label:
        chosen = candidate_chosen
        cleanup_accepted += int(chosen != original_chosen)
    else:
        chosen = original_chosen
        cleanup_fallback += 1

    if parsed_rejected == rejected_label:
        rejected = candidate_rejected
        cleanup_accepted += int(rejected != original_rejected)
    else:
        rejected = original_rejected
        cleanup_fallback += 1

    # Record parser uncertainty/disagreement instead of failing just because
    # the independent parser cannot parse a format Cell 34 already handled.
    final_parse_chosen = parse_answer(chosen)
    final_parse_rejected = parse_answer(rejected)

    if final_parse_chosen is None or final_parse_rejected is None:
        parser_unparsed_after_cleanup += 1

    if (
        final_parse_chosen is not None
        and final_parse_chosen != chosen_label
    ):
        parser_label_disagreements += 1

    if (
        final_parse_rejected is not None
        and final_parse_rejected != rejected_label
    ):
        parser_label_disagreements += 1

    pair["chosen"] = chosen
    pair["rejected"] = rejected
    pair["gold_answer"] = gold
    pair["chosen_answer"] = chosen_label
    pair["rejected_answer"] = rejected_label

    # Both sides must remain parseable for model training, even when no
    # cleanup was safe. An unparsed side is not automatically relabeled.
    if not chosen or not rejected:
        raise ValueError(f"Empty completion after validation at pair {i}.")

    if pair_type == "step_efficiency":
        # The filter preference is valid only if chosen remains shorter.
        if len(chosen) >= len(rejected):
            dropped_efficiency_pairs += 1
            validation_audit.append({
                "pair_index": i,
                "pair_type": pair_type,
                "status": "dropped",
                "reason": "chosen_not_shorter_after_safe_cleanup",
            })
            continue

    training_pairs.append(pair)
    validation_audit.append({
        "pair_index": i,
        "pair_type": pair_type,
        "status": "accepted",
        "gold_answer": gold,
        "chosen_label": chosen_label,
        "rejected_label": rejected_label,
        "chosen_parser_result": final_parse_chosen,
        "rejected_parser_result": final_parse_rejected,
    })


if not training_pairs:
    raise RuntimeError(
        "No preference pairs remain. No GPU training started."
    )

pair_type_counts = {}
for pair in training_pairs:
    kind = pair["pair_type"]
    pair_type_counts[kind] = pair_type_counts.get(kind, 0) + 1

distinct_questions = len({
    str(pair.get("question_index"))
    for pair in training_pairs
})

print("\n" + "=" * 64)
print("USC-DPO — MIXED DATASET VALIDATION")
print("=" * 64)
print("Validated pairs:", len(training_pairs))
print("Answer-correction pairs:", pair_type_counts.get(
    "answer_correction", 0
))
print("Step-efficiency pairs:", pair_type_counts.get(
    "step_efficiency", 0
))
print("Distinct source questions:", distinct_questions)
print("Safe cleanup operations accepted:", cleanup_accepted)
print("Cleanup fallbacks to original text:", cleanup_fallback)
print("Unparsed completions retained using stored labels:",
      parser_unparsed_after_cleanup)
print("Independent parser label disagreements:",
      parser_label_disagreements)
print("Efficiency pairs dropped:", dropped_efficiency_pairs)
print("GPU training has not started yet.")


# Save validation audit before GPU allocation.
audit_path = os.path.join(
    ARTIFACT_DIR, "usc_dpo_mixed_training_validation_audit.json"
)

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(
        validation_audit,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================
# 5. Load the SFT initialization
# =========================================================

for name in (
    "candidate_model",
    "dpo_model",
    "dpo_base_model",
    "dpo_base",
    "dpo_tokenizer",
):
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 64)
print("USC-DPO — MIXED PREFERENCE RETRAINING")
print("=" * 64)
print("Base model:", MODEL_NAME)
print("Initialization: SFT adapter merged into base")
print("Preference pairs:", len(training_pairs))
print("Distinct source questions:", distinct_questions)
print("Epochs:", NUM_EPOCHS)
print("Planned optimizer steps:", NUM_EPOCHS * len(training_pairs))
print("Learning rate:", LEARNING_RATE)
print("DPO beta:", DPO_BETA)
print("LoRA target modules: q_proj, v_proj")
print("GPU:", torch.cuda.get_device_name(0))

dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

dpo_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if dpo_tokenizer.pad_token_id is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

dpo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

sft_initialized = PeftModel.from_pretrained(
    dpo_base,
    SFT_ADAPTER_DIR,
    is_trainable=False,
)

dpo_base_model = sft_initialized.merge_and_unload()

del dpo_base, sft_initialized
gc.collect()
torch.cuda.empty_cache()

dpo_base_model.config.use_cache = False

try:
    dpo_base_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
except TypeError:
    dpo_base_model.gradient_checkpointing_enable()

if hasattr(dpo_base_model, "enable_input_require_grads"):
    dpo_base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

dpo_model = get_peft_model(dpo_base_model, lora_config)

del dpo_base_model
gc.collect()
torch.cuda.empty_cache()

dpo_model.print_trainable_parameters()

trainable_params = [
    parameter
    for parameter in dpo_model.parameters()
    if parameter.requires_grad
]

if not trainable_params:
    raise RuntimeError("No trainable LoRA parameters found.")

device = next(dpo_model.parameters()).device
pad_id = dpo_tokenizer.pad_token_id
eos_id = dpo_tokenizer.eos_token_id


# =========================================================
# 6. Tokenize pairs
# =========================================================

def encode_text(prompt, completion):
    prompt_ids = dpo_tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )["input_ids"]

    completion_ids = dpo_tokenizer(
        completion,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_COMPLETION_TOKENS,
    )["input_ids"]

    if not prompt_ids:
        raise ValueError("Empty prompt after tokenization.")
    if not completion_ids:
        raise ValueError("Empty completion after tokenization.")

    if eos_id is not None and completion_ids[-1] != eos_id:
        completion_ids.append(eos_id)

    return {
        "input_ids": prompt_ids + completion_ids,
        "completion_start": len(prompt_ids),
    }


encoded_pairs = [
    {
        "chosen": encode_text(pair["prompt"], pair["chosen"]),
        "rejected": encode_text(pair["prompt"], pair["rejected"]),
        "question_index": pair.get("question_index"),
        "pair_type": pair["pair_type"],
    }
    for pair in training_pairs
]

print("Pairs encoded:", len(encoded_pairs))


# =========================================================
# 7. Completion log probabilities
# =========================================================

def sequence_log_probability(model, encoded):
    ids = torch.tensor(
        [encoded["input_ids"]],
        dtype=torch.long,
        device=device,
    )
    attention_mask = torch.ones_like(ids)

    outputs = model(
        input_ids=ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

    logits = outputs.logits[:, :-1, :].float()
    labels = ids[:, 1:]

    positions = torch.arange(1, ids.shape[1], device=device)
    completion_mask = (
        positions >= encoded["completion_start"]
    ).unsqueeze(0)

    log_probs = F.log_softmax(logits, dim=-1)
    token_logps = log_probs.gather(
        dim=-1,
        index=labels.unsqueeze(-1),
    ).squeeze(-1)

    score = (token_logps * completion_mask).sum()

    del outputs, logits, log_probs, token_logps
    return score


# =========================================================
# 8. DPO training
# =========================================================

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

global_step = 0
training_history = []

metrics_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_metrics.json"
)
history_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_history.json"
)
pairs_used_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_pairs_used.json"
)

def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (set, tuple)):
        values = list(obj)
        if isinstance(obj, set):
            values = sorted(values, key=str)
        return [json_safe(v) for v in values]
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if hasattr(obj, "item") and callable(obj.item):
        try:
            return obj.item()
        except Exception:
            pass
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def write_metrics():
    metrics = {
        "method": "USC-DPO",
        "base_model": MODEL_NAME,
        "initialization": "SFT adapter merged into base",
        "training_pairs": len(training_pairs),
        "pair_type_counts": pair_type_counts,
        "distinct_source_questions": distinct_questions,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "beta": DPO_BETA,
        "seed": SEED,
        "optimizer_steps": global_step,
        "cleanup_fallbacks": cleanup_fallback,
        "parser_unparsed_after_cleanup": parser_unparsed_after_cleanup,
        "parser_label_disagreements": parser_label_disagreements,
        "lora": {
            "r": 8,
            "alpha": 16,
            "dropout": 0.05,
            "target_modules": ["q_proj", "v_proj"],
        },
        "training_history": training_history,
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(
            json_safe(metrics),
            f,
            ensure_ascii=False,
            indent=2,
        )

    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(
            json_safe(training_history),
            f,
            ensure_ascii=False,
            indent=2,
        )


with open(pairs_used_path, "w", encoding="utf-8") as f:
    json.dump(
        json_safe(training_pairs),
        f,
        ensure_ascii=False,
        indent=2,
    )

for epoch in range(NUM_EPOCHS):
    order = list(range(len(encoded_pairs)))
    random.Random(SEED + epoch).shuffle(order)

    dpo_model.train()

    losses = []
    margins = []
    type_losses = {}
    type_margins = {}

    print(f"\n===== EPOCH {epoch + 1}/{NUM_EPOCHS} =====")

    for step, index in enumerate(order, start=1):
        pair = encoded_pairs[index]
        pair_type = pair["pair_type"]

        dpo_model.eval()

        with torch.no_grad():
            with dpo_model.disable_adapter():
                ref_chosen = sequence_log_probability(
                    dpo_model, pair["chosen"]
                )
                ref_rejected = sequence_log_probability(
                    dpo_model, pair["rejected"]
                )

        dpo_model.train()

        policy_chosen = sequence_log_probability(
            dpo_model, pair["chosen"]
        )
        policy_rejected = sequence_log_probability(
            dpo_model, pair["rejected"]
        )

        policy_ratio = policy_chosen - policy_rejected
        reference_ratio = ref_chosen - ref_rejected

        margin = DPO_BETA * (policy_ratio - reference_ratio)
        loss = -F.logsigmoid(margin)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch + 1}, step {step}."
            )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM,
        )
        optimizer.step()

        global_step += 1

        loss_value = float(loss.detach().cpu())
        margin_value = float(margin.detach().cpu())

        losses.append(loss_value)
        margins.append(margin_value)
        type_losses.setdefault(pair_type, []).append(loss_value)
        type_margins.setdefault(pair_type, []).append(margin_value)

        training_history.append({
            "epoch": epoch + 1,
            "step": global_step,
            "pair_index": step,
            "question_index": pair["question_index"],
            "pair_type": pair_type,
            "loss": loss_value,
            "preference_margin": margin_value,
        })

        if step % LOG_EVERY == 0 or step == len(order):
            print(
                f"Step {step}/{len(order)}"
                f" | loss={loss_value:.4f}"
                f" | margin={margin_value:.4f}",
                flush=True,
            )

        del ref_chosen, ref_rejected
        del policy_chosen, policy_rejected
        del policy_ratio, reference_ratio, margin, loss

    print(f"Epoch mean loss: {sum(losses) / len(losses):.4f}")
    print(f"Epoch mean margin: {sum(margins) / len(margins):.4f}")

    for kind in sorted(type_losses):
        print(
            f"{kind}: "
            f"mean loss={sum(type_losses[kind]) / len(type_losses[kind]):.4f}, "
            f"mean margin={sum(type_margins[kind]) / len(type_margins[kind]):.4f}"
        )

    # Save after each completed epoch.
    dpo_model.save_pretrained(ADAPTER_DIR)
    dpo_tokenizer.save_pretrained(ADAPTER_DIR)
    write_metrics()

    gc.collect()
    torch.cuda.empty_cache()


# =========================================================
# 9. Final artifact save
# =========================================================

dpo_model.save_pretrained(ADAPTER_DIR)
dpo_tokenizer.save_pretrained(ADAPTER_DIR)
write_metrics()

zip_path = "/content/USC_DPO_training_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(ADAPTER_DIR):
        for filename in filenames:
            path = os.path.join(root, filename)
            zf.write(
                path,
                arcname=os.path.relpath(path, ARTIFACT_DIR),
            )

    for path in (
        metrics_path,
        history_path,
        pairs_used_path,
        audit_path,
    ):
        if os.path.isfile(path):
            zf.write(
                path,
                arcname=os.path.basename(path),
            )

print("\n" + "=" * 64)
print("USC-DPO — MIXED PREFERENCE TRAINING FINISHED")
print("=" * 64)
print("Pairs:", len(training_pairs))
print("Pair types:", pair_type_counts)
print("Distinct source questions:", distinct_questions)
print("Epochs:", NUM_EPOCHS)
print("Optimizer steps:", global_step)
print("Final loss:", training_history[-1]["loss"])
print("Adapter:", ADAPTER_DIR)
print("Metrics:", metrics_path)
print("Training pairs used:", pairs_used_path)
print("ZIP:", zip_path)
print("ZIP size (MB):", round(os.path.getsize(zip_path) / 1024**2, 2))

del dpo_model
gc.collect()
torch.cuda.empty_cache()

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("Automatic download unavailable:", repr(exc))
    print("ZIP remains at:", zip_path)


Training answer-parser regression checks: PASS

USC-DPO — MIXED DATASET VALIDATION
Validated pairs: 81
Answer-correction pairs: 61
Step-efficiency pairs: 20
Distinct source questions: 48
Safe cleanup operations accepted: 21
Cleanup fallbacks to original text: 7
Unparsed completions retained using stored labels: 0
Independent parser label disagreements: 0
Efficiency pairs dropped: 0
GPU training has not started yet.

USC-DPO — MIXED PREFERENCE RETRAINING
Base model: Qwen/Qwen2.5-Math-1.5B
Initialization: SFT adapter merged into base
Preference pairs: 81
Distinct source questions: 48
Epochs: 3
Planned optimizer steps: 243
Learning rate: 1e-06
DPO beta: 0.1
LoRA target modules: q_proj, v_proj
GPU: Tesla T4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
Pairs encoded: 81

===== EPOCH 1/3 =====
Step 10/81 | loss=0.6741 | margin=0.0384
Step 20/81 | loss=0.6578 | margin=0.0719
Step 30/81 | loss=0.6945 | margin=-0.0027
Step 40/81 | loss=0.6947 | margin=-0.0032
Step 50/81 | loss=0.6763 | margin=0.0340
Step 60/81 | loss=0.6795 | margin=0.0276
Step 70/81 | loss=0.7159 | margin=-0.0449
Step 80/81 | loss=0.7170 | margin=-0.0472
Step 81/81 | loss=0.6993 | margin=-0.0124
Epoch mean loss: 0.6925
Epoch mean margin: 0.0016
answer_correction: mean loss=0.6926, mean margin=0.0015
step_efficiency: mean loss=0.6925, mean margin=0.0018

===== EPOCH 2/3 =====
Step 10/81 | loss=0.7025 | margin=-0.0186
Step 20/81 | loss=0.6807 | margin=0.0251
Step 30/81 | loss=0.6853 | margin=0.0158
Step 40/81 | loss=0.6585 | margin=0.0705
Step 50/81 | loss=0.7304 | margin=-0.0732
Step 60/81 | loss=0.6795 | margin=0.0275
Step 70/81 | loss=0.7010 | margin=-0.0157
Step 80/81 | loss=0.7176 | margin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# USC-DPO — Recover Final Artifacts Without Retraining
# Reuse the model, training history, and pairs already in memory.

import os
import gc
import json
import zipfile

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
ADAPTER_DIR = os.path.join(ARTIFACT_DIR, "dpo_lora_adapter")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Confirm the completed training state still exists.
required = ["metrics", "training_history", "training_pairs"]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing in-memory training data: {missing}. "
        "Do not rerun training blindly."
    )

# Re-save the final adapter if the trained model is still in memory.
if "dpo_model" in globals():
    dpo_model.save_pretrained(ADAPTER_DIR)

if "dpo_tokenizer" in globals():
    dpo_tokenizer.save_pretrained(ADAPTER_DIR)

adapter_config = os.path.join(ADAPTER_DIR, "adapter_config.json")
if not os.path.isfile(adapter_config):
    raise FileNotFoundError(
        f"Adapter config not found: {adapter_config}"
    )

# Convert sets, tuples, NumPy scalars, and nested values to JSON-safe types.
def to_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (set, tuple)):
        values = list(obj)
        if isinstance(obj, set):
            values = sorted(values, key=str)
        return [to_json_safe(v) for v in values]
    if isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    if hasattr(obj, "item") and callable(obj.item):
        try:
            return obj.item()
        except Exception:
            pass
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)

metrics["training_history"] = training_history
metrics["training_pairs"] = len(training_pairs)
metrics["optimizer_steps"] = len(training_history)

# Record the serialization fix for the saved report.
metrics["artifact_save_recovered"] = True

metrics_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_metrics.json"
)
pairs_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_pairs_used.json"
)
history_path = os.path.join(
    ARTIFACT_DIR, "dpo_training_history.json"
)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        to_json_safe(metrics),
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(pairs_path, "w", encoding="utf-8") as f:
    json.dump(
        to_json_safe(training_pairs),
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(
        to_json_safe(training_history),
        f,
        ensure_ascii=False,
        indent=2,
    )

# Rebuild the ZIP with the adapter and all training records.
zip_path = "/content/USC_DPO_training_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(ADAPTER_DIR):
        for filename in filenames:
            full_path = os.path.join(root, filename)
            zf.write(
                full_path,
                arcname=os.path.relpath(full_path, ARTIFACT_DIR),
            )

    for path in (metrics_path, pairs_path, history_path):
        if os.path.isfile(path):
            zf.write(path, arcname=os.path.basename(path))

print("=" * 60)
print("USC-DPO — ARTIFACT SAVE RECOVERED")
print("=" * 60)
print("Training rerun: NO")
print("Completed optimizer steps:", len(training_history))
print("Final recorded loss:", training_history[-1]["loss"])
print("Adapter config exists:", os.path.isfile(adapter_config))
print("Metrics saved:", metrics_path)
print("Training pairs saved:", pairs_path)
print("Training history saved:", history_path)
print("ZIP saved:", zip_path)
print("ZIP size (MB):", round(os.path.getsize(zip_path) / 1024**2, 2))

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Auto-download unavailable:", repr(e))
    print("ZIP remains at:", zip_path)


USC-DPO — ARTIFACT SAVE RECOVERED
Training rerun: NO
Completed optimizer steps: 216
Final recorded loss: 0.6764364242553711
Adapter config exists: True
Metrics saved: /content/USC_DPO_artifacts/dpo_training_metrics.json
Training pairs saved: /content/USC_DPO_artifacts/dpo_training_pairs_used.json
Training history saved: /content/USC_DPO_artifacts/dpo_training_history.json
ZIP saved: /content/USC_DPO_training_artifacts.zip
ZIP size (MB): 34.72


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# USC-DPO — FINAL EVALUATION
# Compare Base vs SFT-CoT vs USC-DPO on identical held-out GSM8K questions.
# USC-DPO is reconstructed on the merged SFT weights used during training.

import os
import re
import gc
import csv
import json
import time
import zipfile
import torch

from decimal import Decimal
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


# =========================================================
# 1. Configuration and paths
# =========================================================

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
NUM_TEST_SAMPLES = 50
MAX_NEW_TOKENS = 384
SEED = 42

ARTIFACT_DIR = "/content/USC_DPO_artifacts"
SFT_ADAPTER_DIR = os.path.join(ARTIFACT_DIR, "sft_lora_adapter")
DPO_ADAPTER_DIR = os.path.join(ARTIFACT_DIR, "dpo_lora_adapter")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. Enable GPU in Colab.")

for path in (SFT_ADAPTER_DIR, DPO_ADAPTER_DIR):
    if not os.path.isfile(os.path.join(path, "adapter_config.json")):
        raise FileNotFoundError(
            f"Adapter not found or incomplete: {path}"
        )

eval_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("=" * 64)
print("USC-DPO — FINAL EVALUATION")
print("=" * 64)
print("Base model:", MODEL_NAME)
print("Test examples:", NUM_TEST_SAMPLES)
print("Max new tokens:", MAX_NEW_TOKENS)
print("GPU:", torch.cuda.get_device_name(0))
print("SFT adapter:", SFT_ADAPTER_DIR)
print("USC-DPO adapter:", DPO_ADAPTER_DIR)


# =========================================================
# 2. Load the held-out test questions and tokenizer
# =========================================================

if "gsm8k" not in globals():
    gsm8k = load_dataset("openai/gsm8k", "main")

eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if eval_tokenizer.pad_token_id is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_questions = [
    gsm8k["test"][i]
    for i in range(min(NUM_TEST_SAMPLES, len(gsm8k["test"])))
]

if not eval_questions:
    raise RuntimeError("GSM8K test set is empty.")

print("Loaded test questions:", len(eval_questions))


# =========================================================
# 3. Answer extraction
# =========================================================

def normalize_numeric_answer(value):
    if value is None:
        return None

    value = str(value).strip().replace(",", "").replace("$", "")

    try:
        number = Decimal(value)

        if not number.is_finite():
            return None

        if number == number.to_integral_value():
            return str(int(number))

        return format(number.normalize(), "f")

    except Exception:
        return None


def extract_boxed_answer(text):
    """
    Extract the content of \\boxed{...}, including nested braces,
    e.g. \\boxed{\\color{blue}{45}}.
    """
    for match in re.finditer(r"\\boxed\s*\{", text):
        start = match.end()
        depth = 1
        pos = start

        while pos < len(text) and depth > 0:
            if text[pos] == "{":
                depth += 1
            elif text[pos] == "}":
                depth -= 1
            pos += 1

        if depth == 0:
            content = text[start:pos - 1]
            numbers = re.findall(
                r"[-+]?\d[\d,]*(?:\.\d+)?",
                content,
            )

            if numbers:
                answer = normalize_numeric_answer(numbers[-1])
                if answer is not None:
                    return answer

    return None


def extract_prediction(text):
    # Priority 1: explicit GSM8K answer markers.
    matches = re.findall(
        r"####\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        text,
    )

    if matches:
        return normalize_numeric_answer(matches[-1])

    # Priority 2: boxed answers, including nested LaTeX formatting.
    boxed = extract_boxed_answer(text)

    if boxed is not None:
        return boxed

    # Priority 3: explicit answer phrases.
    patterns = [
        r"\bfinal answer\s*(?:is|:|=)?\s*\$?\s*"
        r"([-+]?\d[\d,]*(?:\.\d+)?)",
        r"\bthe answer is\s*\$?\s*"
        r"([-+]?\d[\d,]*(?:\.\d+)?)",
        r"\banswer\s*:\s*\$?\s*"
        r"([-+]?\d[\d,]*(?:\.\d+)?)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        if matches:
            return normalize_numeric_answer(matches[-1])

    # Priority 4: last number on the last line with a number.
    # This is a fallback and can still misparse ambiguous responses.
    for line in reversed(text.splitlines()):
        numbers = re.findall(
            r"[-+]?\d[\d,]*(?:\.\d+)?",
            line,
        )

        if numbers:
            return normalize_numeric_answer(numbers[-1])

    return None


def extract_gold(example):
    match = re.search(
        r"####\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        example["answer"],
    )

    return (
        normalize_numeric_answer(match.group(1))
        if match
        else None
    )


def make_eval_prompt(question):
    # Identical instruction and decoding configuration for every model.
    return (
        "Solve this math word problem carefully. "
        "Show concise reasoning and the key calculations.\n"
        "End with exactly one final line in this format: #### <number>\n"
        "Do not write anything after the final-answer line.\n\n"
        f"Problem:\n{question}\n\n"
        "Solution:\n"
    )


# =========================================================
# 4. Load and evaluate one variant at a time
# =========================================================

def evaluate_variant(variant_name):
    print("\n" + "=" * 64)
    print("Evaluating:", variant_name)
    print("=" * 64, flush=True)

    # Start from the same base checkpoint for each variant.
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=eval_dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    if variant_name == "Base":
        eval_model = base

    elif variant_name == "SFT-CoT":
        eval_model = PeftModel.from_pretrained(
            base,
            SFT_ADAPTER_DIR,
            is_trainable=False,
        )

    elif variant_name == "USC-DPO":
        # Training procedure was:
        # original base -> load SFT adapter -> merge SFT weights
        # -> attach/train a new USC-DPO adapter.
        #
        # Reconstruct exactly that initialization at evaluation.
        sft_initialized = PeftModel.from_pretrained(
            base,
            SFT_ADAPTER_DIR,
            is_trainable=False,
        )

        merged_sft = sft_initialized.merge_and_unload()

        del sft_initialized
        del base
        gc.collect()
        torch.cuda.empty_cache()

        eval_model = PeftModel.from_pretrained(
            merged_sft,
            DPO_ADAPTER_DIR,
            is_trainable=False,
        )

        # eval_model now owns the merged SFT backbone and USC-DPO adapter.
        base = merged_sft

    else:
        del base
        raise ValueError(f"Unknown evaluation variant: {variant_name}")

    eval_model.eval()
    device = next(eval_model.parameters()).device

    variant_results = []
    correct_count = 0
    start_variant = time.time()

    try:
        for i, example in enumerate(eval_questions, start=1):
            question = example["question"]
            gold = extract_gold(example)
            prompt = make_eval_prompt(question)

            inputs = eval_tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
            ).to(device)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.time()

            with torch.inference_mode():
                output_ids = eval_model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    repetition_penalty=1.05,
                    pad_token_id=eval_tokenizer.pad_token_id,
                    eos_token_id=eval_tokenizer.eos_token_id,
                    use_cache=True,
                )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            latency = time.time() - start

            new_tokens = output_ids[
                0, inputs["input_ids"].shape[1]:
            ]

            generated_text = eval_tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            ).strip()

            predicted = extract_prediction(generated_text)

            correct = (
                predicted is not None
                and gold is not None
                and predicted == gold
            )

            correct_count += int(correct)

            row = {
                "model": variant_name,
                "test_index": i - 1,
                "question": question,
                "prediction": predicted,
                "gold": gold,
                "correct": correct,
                "latency_seconds": round(latency, 4),
                "generated_tokens": int(new_tokens.numel()),
                "generated_text": generated_text,
            }

            variant_results.append(row)

            if i % 5 == 0 or i == len(eval_questions):
                print(
                    f"[{i}/{len(eval_questions)}] "
                    f"Accuracy so far: {correct_count}/{i} "
                    f"({100 * correct_count / i:.1f}%) | "
                    f"Last prediction={predicted} | Gold={gold}",
                    flush=True,
                )

            del inputs, output_ids, new_tokens

    finally:
        # Release GPU model memory, including on evaluation errors.
        del eval_model
        del base
        gc.collect()
        torch.cuda.empty_cache()

    total_count = len(variant_results)

    summary = {
        "model": variant_name,
        "correct": correct_count,
        "total": total_count,
        "unparsed_predictions": sum(
            row["prediction"] is None
            for row in variant_results
        ),
        "accuracy_percent": (
            100.0 * correct_count / total_count
            if total_count else 0.0
        ),
        "avg_latency_seconds": (
            sum(row["latency_seconds"] for row in variant_results)
            / total_count
            if total_count else 0.0
        ),
        "avg_generated_tokens": (
            sum(row["generated_tokens"] for row in variant_results)
            / total_count
            if total_count else 0.0
        ),
        "elapsed_seconds": round(time.time() - start_variant, 2),
    }

    print("\nSummary:", summary, flush=True)
    return summary, variant_results


# =========================================================
# 5. Run all variants on the exact same test examples
# =========================================================

all_summaries = []
all_predictions = []

for variant in ("Base", "SFT-CoT", "USC-DPO"):
    summary, rows = evaluate_variant(variant)
    all_summaries.append(summary)
    all_predictions.extend(rows)

# Ensure all models have results for the same test indices.
expected_rows = len(eval_questions) * 3

if len(all_predictions) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} prediction rows, "
        f"received {len(all_predictions)}."
    )


# =========================================================
# 6. Print comparison
# =========================================================

print("\n" + "=" * 82)
print("FINAL COMPARISON — GSM8K TEST")
print("=" * 82)

print(
    f"{'Model':<14}"
    f"{'Correct':>12}"
    f"{'Accuracy':>13}"
    f"{'Latency':>13}"
    f"{'Tokens':>12}"
)

for row in all_summaries:
    print(
        f"{row['model']:<14}"
        f"{str(row['correct']) + '/' + str(row['total']):>12}"
        f"{row['accuracy_percent']:>12.2f}%"
        f"{row['avg_latency_seconds']:>11.2f}s"
        f"{row['avg_generated_tokens']:>12.1f}"
    )

print(
    "\nNote: this is a 50-question evaluation subset, "
    "not a statistically conclusive benchmark."
)


# =========================================================
# 7. Save metrics, predictions, CSV and reproducible ZIP
# =========================================================

summary_path = os.path.join(
    ARTIFACT_DIR, "final_evaluation_summary.json"
)

predictions_path = os.path.join(
    ARTIFACT_DIR, "final_evaluation_predictions.json"
)

csv_path = os.path.join(
    ARTIFACT_DIR, "final_evaluation_comparison.csv"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(all_summaries, f, ensure_ascii=False, indent=2)

with open(predictions_path, "w", encoding="utf-8") as f:
    json.dump(all_predictions, f, ensure_ascii=False, indent=2)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    fields = [
        "model",
        "correct",
        "total",
        "unparsed_predictions",
        "accuracy_percent",
        "avg_latency_seconds",
        "avg_generated_tokens",
        "elapsed_seconds",
    ]

    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(all_summaries)

zip_path = "/content/USC_DPO_final_evaluation.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in (summary_path, predictions_path, csv_path):
        zf.write(path, arcname=os.path.basename(path))

    for adapter_dir in (SFT_ADAPTER_DIR, DPO_ADAPTER_DIR):
        for root, _, filenames in os.walk(adapter_dir):
            for filename in filenames:
                full_path = os.path.join(root, filename)
                zf.write(
                    full_path,
                    arcname=os.path.relpath(full_path, ARTIFACT_DIR),
                )

print("\nEvaluation saved:")
print("Summary:", summary_path)
print("Per-question predictions:", predictions_path)
print("CSV comparison:", csv_path)
print("ZIP:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Auto-download unavailable:", repr(e))
    print("ZIP remains at:", zip_path)


USC-DPO — FINAL EVALUATION
Base model: Qwen/Qwen2.5-Math-1.5B
Test examples: 50
Max new tokens: 384
GPU: Tesla T4
SFT adapter: /content/USC_DPO_artifacts/sft_lora_adapter
USC-DPO adapter: /content/USC_DPO_artifacts/dpo_lora_adapter
Loaded test questions: 50

Evaluating: Base


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[5/50] Accuracy so far: 3/5 (60.0%) | Last prediction=20 | Gold=20
[10/50] Accuracy so far: 5/10 (50.0%) | Last prediction=460 | Gold=460
[15/50] Accuracy so far: 9/15 (60.0%) | Last prediction=60 | Gold=60
[20/50] Accuracy so far: 11/20 (55.0%) | Last prediction=4 | Gold=6
[25/50] Accuracy so far: 14/25 (56.0%) | Last prediction=26 | Gold=26
[30/50] Accuracy so far: 17/30 (56.7%) | Last prediction=94 | Gold=104
[35/50] Accuracy so far: 21/35 (60.0%) | Last prediction=23 | Gold=23
[40/50] Accuracy so far: 23/40 (57.5%) | Last prediction=18 | Gold=18
[45/50] Accuracy so far: 26/45 (57.8%) | Last prediction=20 | Gold=20
[50/50] Accuracy so far: 30/50 (60.0%) | Last prediction=30 | Gold=30

Summary: {'model': 'Base', 'correct': 30, 'total': 50, 'unparsed_predictions': 0, 'accuracy_percent': 60.0, 'avg_latency_seconds': 7.719048, 'avg_generated_tokens': 209.88, 'elapsed_seconds': 386.35}

Evaluating: SFT-CoT


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[5/50] Accuracy so far: 4/5 (80.0%) | Last prediction=20 | Gold=20
[10/50] Accuracy so far: 6/10 (60.0%) | Last prediction=460 | Gold=460
[15/50] Accuracy so far: 10/15 (66.7%) | Last prediction=60 | Gold=60
[20/50] Accuracy so far: 12/20 (60.0%) | Last prediction=4 | Gold=6
[25/50] Accuracy so far: 15/25 (60.0%) | Last prediction=26 | Gold=26
[30/50] Accuracy so far: 18/30 (60.0%) | Last prediction=99 | Gold=104
[35/50] Accuracy so far: 23/35 (65.7%) | Last prediction=23 | Gold=23
[40/50] Accuracy so far: 25/40 (62.5%) | Last prediction=18 | Gold=18
[45/50] Accuracy so far: 28/45 (62.2%) | Last prediction=20 | Gold=20
[50/50] Accuracy so far: 32/50 (64.0%) | Last prediction=30 | Gold=30

Summary: {'model': 'SFT-CoT', 'correct': 32, 'total': 50, 'unparsed_predictions': 0, 'accuracy_percent': 64.0, 'avg_latency_seconds': 10.143008, 'avg_generated_tokens': 212.42, 'elapsed_seconds': 507.56}

Evaluating: USC-DPO


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[5/50] Accuracy so far: 3/5 (60.0%) | Last prediction=20 | Gold=20
[10/50] Accuracy so far: 5/10 (50.0%) | Last prediction=460 | Gold=460
[15/50] Accuracy so far: 9/15 (60.0%) | Last prediction=60 | Gold=60
[20/50] Accuracy so far: 11/20 (55.0%) | Last prediction=4 | Gold=6
[25/50] Accuracy so far: 13/25 (52.0%) | Last prediction=26 | Gold=26
[30/50] Accuracy so far: 16/30 (53.3%) | Last prediction=94 | Gold=104
[35/50] Accuracy so far: 20/35 (57.1%) | Last prediction=23 | Gold=23
[40/50] Accuracy so far: 23/40 (57.5%) | Last prediction=18 | Gold=18
[45/50] Accuracy so far: 26/45 (57.8%) | Last prediction=20 | Gold=20
[50/50] Accuracy so far: 30/50 (60.0%) | Last prediction=30 | Gold=30

Summary: {'model': 'USC-DPO', 'correct': 30, 'total': 50, 'unparsed_predictions': 0, 'accuracy_percent': 60.0, 'avg_latency_seconds': 10.092666000000001, 'avg_generated_tokens': 212.6, 'elapsed_seconds': 505.05}

FINAL COMPARISON — GSM8K TEST
Model              Correct     Accuracy      Latency      To

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>